# DI 725 Phase-3 - Cross-modal Fusion Ablations

Aras400K + frozen RemoteCLIP. Phase-2 established the four-fusion comparison;
Phase-3 adds three ablations: tau threshold sensitivity, full segmentation matrix,
and a ViT-B/16 backbone.

Repo:
<https://github.com/sceran/aras-multimodal-attention-landcover-classification>


## Setup


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!pip install -q open_clip_torch huggingface_hub wandb scikit-learn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.8 MB/s eta 0:00:00


In [ ]:
import wandb
wandb.login()


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: sceran to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## 01 Tau Ablation

Threshold (tau) sensitivity ablation
Sweep: 4 fusion x 5 caption x 3 tau x 3 seed = 180 runs

RemoteCLIP-ViT-B/32 patch features). Outputs a single JSON aggregating all
runs by (fusion, caption, tau).




In [ ]:
# 1 - Config, load features, captions, build label tensors for each tau
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import average_precision_score, f1_score
from sklearn.model_selection import train_test_split

CONFIG = {
    "device":        "cuda",
    "data_root":     Path("/content/drive/MyDrive/Colab Notebooks/DI725/DI725_project_dataset"),
    "feat_dir":      Path("/content/drive/MyDrive/Colab Notebooks/DI725/phase2_features"),
    "results_dir":   Path("/content/drive/MyDrive/Colab Notebooks/DI725/phase3_results"),
    "patch_file":    "patch_features_remoteclip_vitb32.pt",
    "split_seed":    42,
    "val_frac":      0.2,
    "seeds":         [42, 1337, 2024],
    "taus":          [5, 10, 20],
    "fusions":       ["image_only", "late", "film", "gated", "cross_attn"],
    "captions":      ["hybrid_gemma3-4b", "hybrid_qwen3-vl-8b", "text_qwen3-4b",
                      "vision_gemma3-4b", "vision_qwen3-vl-8b"],
    "classes":       ["Tree", "Shrub", "Grass", "Crop", "Built-up", "Barren", "Water"],
    "epochs":        30,
    "batch_size":    256,
    "lr":            1e-3,
    "weight_decay":  1e-4,
    "feature_dim":   512,
    "hidden":        256,
    "n_heads":       8,
    "dropout":       0.1,
    "wandb_project": "di725-phase3-tau",
}
CONFIG["results_dir"].mkdir(parents=True, exist_ok=True)

DEVICE  = CONFIG["device"]
CLASSES = CONFIG["classes"]
N_CLASSES = len(CLASSES)

# Patch features (10000, 50, 512) FP16; pos 0 is CLS, 1..49 are patches.
patch_data = torch.load(CONFIG["feat_dir"] / CONFIG["patch_file"], map_location="cpu")
patch_features = patch_data["patch_features"]
filenames_cached = patch_data["filenames"]
print(f"Patch features: {tuple(patch_features.shape)}  dtype={patch_features.dtype}")

df = pd.read_csv(CONFIG["data_root"] / "captions.csv")
assert df["filename"].tolist() == filenames_cached, "captions.csv order differs from cached features"

# Fixed 80/20 stratified split on dominant class (same across all conditions).
dominant = df[CLASSES].values.argmax(1)
train_idx, val_idx = train_test_split(
    np.arange(len(df)), test_size=CONFIG["val_frac"],
    random_state=CONFIG["split_seed"], stratify=dominant,
)
print(f"Train: {len(train_idx)}, Val: {len(val_idx)}")

# Pre-build label tensors per tau so the training loop just indexes by tau.
composition = df[CLASSES].values.astype(np.float32)
labels_by_tau = {}
for tau in CONFIG["taus"]:
    y = (composition >= tau).astype(np.float32)
    labels_by_tau[tau] = torch.tensor(y, device=DEVICE)
    pos_per_class = dict(zip(CLASSES, y.sum(0).astype(int).tolist()))
    print(f"tau={tau:>2}%  positives/class: {pos_per_class}")

# Features on GPU once.
patch_features_gpu = patch_features.to(DEVICE)
image_cls_gpu     = patch_features_gpu[:, 0, :]
image_patches_gpu = patch_features_gpu[:, 1:, :]
print(f"image_cls={tuple(image_cls_gpu.shape)}  image_patches={tuple(image_patches_gpu.shape)}")



Patch features: (10000, 50, 512)  dtype=torch.float16
Train: 8000, Val: 2000
tau= 5%  positives/class: {'Tree': 5369, 'Shrub': 491, 'Grass': 8567, 'Crop': 3606, 'Built-up': 473, 'Barren': 1864, 'Water': 458}
tau=10%  positives/class: {'Tree': 4769, 'Shrub': 296, 'Grass': 7839, 'Crop': 3126, 'Built-up': 256, 'Barren': 1060, 'Water': 378}
tau=20%  positives/class: {'Tree': 4126, 'Shrub': 127, 'Grass': 6743, 'Crop': 2616, 'Built-up': 138, 'Barren': 538, 'Water': 290}
image_cls=(10000, 512)  image_patches=(10000, 49, 512)


In [ ]:
# 2 - Hook-based text encoder, cache the 5 caption sets
import open_clip
from huggingface_hub import hf_hub_download

if "model" not in globals():
    print("Loading RemoteCLIP-ViT-B/32...")
    ckpt_path = hf_hub_download("chendelong/RemoteCLIP", "RemoteCLIP-ViT-B-32.pt")
    model, _, _ = open_clip.create_model_and_transforms("ViT-B-32")
    model.load_state_dict(torch.load(ckpt_path, map_location="cpu"))
    model = model.to(DEVICE).eval()
    tokenizer = open_clip.get_tokenizer("ViT-B-32")
else:
    print("Model already loaded.")


@torch.no_grad()
def encode_text_tokens_and_pooled(captions, batch_size=256):
    all_tokens, all_pooled = [], []
    for i in range(0, len(captions), batch_size):
        batch = captions[i : i + batch_size]
        text_ids = tokenizer(batch).to(DEVICE)

        captured = {}
        def hook(_, __, out):
            captured["tokens"] = out

        h = model.ln_final.register_forward_hook(hook)
        try:
            pooled = model.encode_text(text_ids)
        finally:
            h.remove()

        tokens = captured["tokens"] @ model.text_projection
        tokens = tokens / tokens.norm(dim=-1, keepdim=True)
        pooled = pooled / pooled.norm(dim=-1, keepdim=True)
        all_tokens.append(tokens.cpu().half())
        all_pooled.append(pooled.cpu().half())
    return torch.cat(all_tokens), torch.cat(all_pooled)


text_tokens = {}
text_pooled = {}
for col in CONFIG["captions"]:
    print(f"Encoding {col}...")
    captions = df[col].fillna("").tolist()
    tk, pl = encode_text_tokens_and_pooled(captions)
    text_tokens[col] = tk
    text_pooled[col] = pl
    print(f"  tokens={tuple(tk.shape)} pooled={tuple(pl.shape)}")



Loading RemoteCLIP-ViT-B/32...


RemoteCLIP-ViT-B-32.pt:   0%|          | 0.00/605M [00:00<?, ?B/s]

Encoding hybrid_gemma3-4b...
  tokens=(10000, 77, 512) pooled=(10000, 512)
Encoding hybrid_qwen3-vl-8b...
  tokens=(10000, 77, 512) pooled=(10000, 512)
Encoding text_qwen3-4b...
  tokens=(10000, 77, 512) pooled=(10000, 512)
Encoding vision_gemma3-4b...
  tokens=(10000, 77, 512) pooled=(10000, 512)
Encoding vision_qwen3-vl-8b...
  tokens=(10000, 77, 512) pooled=(10000, 512)


In [ ]:
# 3- Fusion modules (dim-parameterised so the same class definitions
# work for B/32 features (dim=512) and the L/14 ablation (dim=768))
DIM = CONFIG["feature_dim"]   # default 512 (B/32); pass dim at construction to override
H   = CONFIG["hidden"]
DR  = CONFIG["dropout"]
NH  = CONFIG["n_heads"]


class ImageOnlyHead(nn.Module):
    def __init__(self, dim=DIM, hidden=H, dropout=DR):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, N_CLASSES),
        )
    def forward(self, image_cls):
        return self.net(image_cls)


class LateFusion(nn.Module):
    def __init__(self, dim=DIM, hidden=H, dropout=DR):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim * 2, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, N_CLASSES),
        )
    def forward(self, image_cls, text_pooled):
        return self.net(torch.cat([image_cls, text_pooled], dim=-1))


class FiLMFusion(nn.Module):
    """Text -> (gamma, beta) modulating image_cls. Final layer zero-init so
    gamma starts at 1 and beta at 0 (identity at step 0)."""
    def __init__(self, dim=DIM, hidden=H, dropout=DR):
        super().__init__()
        self.modulator = nn.Sequential(
            nn.Linear(dim, hidden), nn.ReLU(),
            nn.Linear(hidden, dim * 2),
        )
        nn.init.zeros_(self.modulator[-1].weight)
        nn.init.zeros_(self.modulator[-1].bias)
        self.head = nn.Sequential(
            nn.Linear(dim, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, N_CLASSES),
        )
    def forward(self, image_cls, text_pooled):
        gamma, beta = self.modulator(text_pooled).chunk(2, dim=-1)
        return self.head((1.0 + gamma) * image_cls + beta)


class GatedFusion(nn.Module):
    def __init__(self, dim=DIM, hidden=H, dropout=DR):
        super().__init__()
        self.gate = nn.Sequential(nn.Linear(dim * 2, dim), nn.Sigmoid())
        self.head = nn.Sequential(
            nn.Linear(dim, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, N_CLASSES),
        )
    def forward(self, image_cls, text_pooled):
        g = self.gate(torch.cat([image_cls, text_pooled], dim=-1))
        return self.head(g * image_cls + (1.0 - g) * text_pooled)


class CrossAttentionFusion(nn.Module):
    """Text tokens (Q) attend over image patches (K, V); mean-pool then classify."""
    def __init__(self, dim=DIM, n_heads=NH, hidden=H, dropout=DR):
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, n_heads, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Sequential(
            nn.Linear(dim, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, N_CLASSES),
        )
    def forward(self, image_patches, text_tokens_in):
        attended, _ = self.attn(query=text_tokens_in, key=image_patches, value=image_patches)
        return self.head(self.norm(attended + text_tokens_in).mean(dim=1))


MODULES = {
    "image_only": ImageOnlyHead,
    "late":       LateFusion,
    "film":       FiLMFusion,
    "gated":      GatedFusion,
    "cross_attn": CrossAttentionFusion,
}


def features_for(condition, caption_col):
    if condition == "image_only":
        return (image_cls_gpu.float(),)
    if condition in ("late", "film", "gated"):
        return (image_cls_gpu.float(), text_pooled[caption_col].to(DEVICE).float())
    if condition == "cross_attn":
        return (image_patches_gpu.float(), text_tokens[caption_col].to(DEVICE).float())
    raise ValueError(condition)



In [ ]:
# 4 - Training helper
import random as _random

import wandb


def set_seeds(seed):
    _random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def train_one(condition, caption_col, tau, seed):
    set_seeds(seed)
    name = f"{condition}__{caption_col or 'none'}__tau{tau}__s{seed}"

    feats = features_for(condition, caption_col)
    net   = MODULES[condition]().to(DEVICE)
    opt   = torch.optim.AdamW(net.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
    loss_fn = nn.BCEWithLogitsLoss()

    y_all = labels_by_tau[tau]
    yt    = y_all[train_idx]
    yv_np = y_all[val_idx].cpu().numpy()

    wandb.init(
        project=CONFIG["wandb_project"], name=name, reinit=True,
        tags=[f"fusion={condition}", f"caption={caption_col or 'none'}",
              f"tau={tau}", f"seed={seed}"],
        config={"condition": condition, "caption": caption_col, "tau": tau,
                "seed": seed, "epochs": CONFIG["epochs"], "lr": CONFIG["lr"],
                "wd": CONFIG["weight_decay"], "batch_size": CONFIG["batch_size"]},
    )

    best = {"val_mAP": 0.0, "val_f1_macro": 0.0, "val_f1_per_class": None, "epoch": -1}
    tr_idx_t = torch.tensor(train_idx, device=DEVICE)
    v_idx_t  = torch.tensor(val_idx,  device=DEVICE)
    v_inputs = tuple(f[v_idx_t] for f in feats)

    for ep in range(CONFIG["epochs"]):
        net.train()
        perm = torch.randperm(len(train_idx), device=DEVICE)
        epoch_loss = 0.0; nbatch = 0
        for i in range(0, len(train_idx), CONFIG["batch_size"]):
            b = perm[i : i + CONFIG["batch_size"]]
            inputs = tuple(f[tr_idx_t[b]] for f in feats)
            opt.zero_grad()
            loss = loss_fn(net(*inputs), yt[b])
            loss.backward(); opt.step()
            epoch_loss += loss.item(); nbatch += 1

        net.eval()
        with torch.no_grad():
            logits = net(*v_inputs).cpu().numpy()
        probs  = 1.0 / (1.0 + np.exp(-logits))
        preds  = (probs > 0.5).astype(int)
        mAP    = average_precision_score(yv_np, probs, average="macro")
        f1m    = f1_score(yv_np, preds, average="macro", zero_division=0)
        f1pc   = f1_score(yv_np, preds, average=None,    zero_division=0)

        log = {"epoch": ep, "train/loss": epoch_loss / nbatch,
               "val/mAP": mAP, "val/f1_macro": f1m}
        for c, f in zip(CLASSES, f1pc):
            log[f"val/f1_{c}"] = f
        wandb.log(log)

        if mAP > best["val_mAP"]:
            best = {"val_mAP": float(mAP), "val_f1_macro": float(f1m),
                    "val_f1_per_class": [float(x) for x in f1pc], "epoch": ep}

    wandb.finish()
    return best



In [ ]:
# 5 - Sweep: 4 fusion x 5 caption x 3 tau x 3 seed (image_only x 3 tau x 3 seed)
results = {}   # key = (fusion, caption_or_none, tau)  ->  list of best dicts
total = 0

for tau in CONFIG["taus"]:
    print(f"\n##### TAU = {tau}% #####")
    for seed in CONFIG["seeds"]:
        # image-only baseline once per tau-seed
        key = ("image_only", "none", tau)
        best = train_one("image_only", None, tau, seed)
        results.setdefault(key, []).append(best)
        total += 1

        for cap in CONFIG["captions"]:
            for fusion in ("late", "film", "gated", "cross_attn"):
                key = (fusion, cap, tau)
                best = train_one(fusion, cap, tau, seed)
                results.setdefault(key, []).append(best)
                total += 1

print(f"\nTotal runs: {total}")




##### TAU = 5% #####


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train/loss,█▅▄▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/f1_Barren,▁▁▄▄▅▆▆▆▇▇▇██▇▇▇█▇▇███████████
val/f1_Built-up,▁▁▁▁▁▁▁▂▄▅▆▆▆▇▇▇▇▇▇▇▇▇▇███████
val/f1_Crop,▁▇▇███████████████████████████
val/f1_Grass,▁▁▁▁▂▂▄▅▆▆▇▇▇▇▇▇▇▇██▇█▇███████
val/f1_Shrub,▁▁▁▁▁▁▁▁▁▁▁▂▂▂▁▁▁▄▅▂▂▅▅▄▅▆▄▇▄█
val/f1_Tree,▁▄▄▄▅▅▅▆▆▆▆▇▇▇▇▇▇▇▇███████████
val/f1_Water,▁▁▁▁▁▁▄▆▇▇▇███████████████████
val/f1_macro,▁▂▃▃▄▄▅▅▆▆▇▇▇▇▇▇▇▇█▇▇█████████
+1,...


epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train/loss,█▅▄▄▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/f1_Barren,▁▁▄▃▅▆▇▇▇▇▇▇▇▇▇██▇████████████
val/f1_Built-up,▁▁▁▁▁▁▁▂▃▄▆▆▆▇▇▇▇▇▇▇██████████
val/f1_Crop,▁▇▇▇██████████████████████████
val/f1_Grass,▁▁▁▁▂▄▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇████████
val/f1_Shrub,▁▁▁▁▁▁▁▁▁▁▁▃▄▄▄▄▅▆▆▆▆▆▆▇▇▇▇█▇█
val/f1_Tree,▁▂▂▃▄▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇▇███▇█████
val/f1_Water,▁▁▁▁▁▄▆▇▇▇▇▇▇▇████████████████
val/f1_macro,▁▂▂▂▃▄▅▅▅▆▆▆▆▇▇▇▇▇▇▇▇▇████████
+1,...


epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train/loss,█▅▄▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/f1_Barren,▁▁▂▆▆▇▇▇▆▇████████████████████
val/f1_Built-up,▁▁▁▃▅▆▇▇▇███████████████████▇█
val/f1_Crop,▁▇▇▇██████████████████████████
val/f1_Grass,▁▁▂▄▄▅▄▆▆▆▆▇▇▇▇▇▇▇▇█▇██▇██████
val/f1_Shrub,▁▁▁▁▃▅▅▅▆▆▆▇▇▇▇▇▇▇██████▇█▇███
val/f1_Tree,▁▂▄▅▅▆▆▆▇▇▆▇▇▇▇▇▇▇▇▇▇████████▇
val/f1_Water,▁▁▆▇▇█████████████████████████
val/f1_macro,▁▂▃▅▆▇▇▇▇▇▇███████████████████
+1,...


...

epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train/loss,█▄▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/f1_Barren,▁▁▁▁▁▁▃▄▆▅▆▆▇▇▇▇▇▇▇▇▇█▇▇▇████▇
val/f1_Built-up,▁▁▁▁▁▁▁▁▁▁▁▁▁▂▂▃▃▄▅▅▅▆▇▇▇▇▇███
val/f1_Crop,▁▆▇▇▇█████████████████████████
val/f1_Grass,▁▁▂▄▅▆▇▇▇▇▇▇█▇██████▇██▇██████
val/f1_Shrub,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/f1_Tree,▁▄▄▅▅▆▆▇▇▇▇▇▇▇▇█▇▇▇███████████
val/f1_Water,▁▁▁▁▁▂▄▆▇█████████████████████
val/f1_macro,▁▂▃▃▃▃▄▅▆▆▆▆▆▆▆▇▇▇▇▇▇█████████
+1,...


epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train/loss,█▅▄▄▃▃▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁
val/f1_Barren,▁▁▁▁▃▆▇▆▆▇█▇▇▇▇▇█▇█▇█▇▇▇▇▇██▇█
val/f1_Built-up,▁▁▁▁▁▂▇▇▇▇███▇██▇██████▇█████▇
val/f1_Crop,▁▆▇▇▇█████████████████████████
val/f1_Grass,▁▁▃▄▆▆▆▇▇▇█▇██▇▇██▇██▇███▇████
val/f1_Shrub,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▃▃█▁▄▃▁▆▄▄▆▁
val/f1_Tree,▁▇▇▇▇█████████████████████████
val/f1_Water,▁▁▁▃▇▇████████████████████████
val/f1_macro,▁▃▃▄▅▆▇▇▇▇█▇▇▇█▇▇▇▇██▇█▇▇█████
+1,...


epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train/loss,█▅▄▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/f1_Barren,▁▁▁▁▁▁▁▁▁▂▄▅▆▆▆▇▇▇▇▇▇▇████████
val/f1_Built-up,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▂▃▃▃▅▆▆▆▇▇▇████
val/f1_Crop,▁▁▆▇▇▇▇███████████████████████
val/f1_Grass,▁▁▁▁▂▄▅▆▇▇▇▇▇▇████████████████
val/f1_Shrub,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/f1_Tree,▁▇▇▇▇█████████████████████████
val/f1_Water,▁▁▁▁▁▁▁▂▃▇████████████████████
val/f1_macro,▁▂▄▄▄▄▄▄▅▆▆▆▇▇▇▇▇▇▇▇▇█████████
+1,...


epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train/loss,█▄▄▃▃▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁
val/f1_Barren,▁▆▇█▆█▇████▇███▇███▇█▇████████
val/f1_Built-up,▁▅▇▇▇▇█▇███▇█▇██▇▇▇▇███▇██▇███
val/f1_Crop,▁▃▄▅▆▄▆▇▇▇▆▄██▇█▇▆▅▆█▇▆▇▇▇▇▇▇▆
val/f1_Grass,▁▂▆▆▇▇▆██▆▇▇▇▇▅█▇▄█▅▆▇▇███▆▆▇▇
val/f1_Shrub,▁▁▁▁▁▁▁▂▁▁▆▅▇▅▄▁▅█▆▇▃▇▃▂▃▁▃▅▂▇
val/f1_Tree,▁▅▄▆▇▇▆▆▇▅▆█▇▇▇▇▇▄▇▇▆▆▆▅▅▅▃▁▄▂
val/f1_Water,▁▇█▇██████▇█▇▇▇██▇█▇▇█▇██▇▇▅▇▇
val/f1_macro,▁▅▆▇▆▇▇▇▇▇█▇██▇▇███▇▇█▇▇▇▇▇▇▇█
+1,...



Total runs: 189


In [ ]:
# 6 - Aggregate (mean/std across seeds), save JSON, print headline tables
agg = {}
for (fusion, cap, tau), runs in results.items():
    mAPs  = np.array([r["val_mAP"] for r in runs])
    f1ms  = np.array([r["val_f1_macro"] for r in runs])
    f1pcs = np.array([r["val_f1_per_class"] for r in runs])
    agg[f"{fusion}__{cap}__tau{tau}"] = {
        "fusion":  fusion,
        "caption": cap,
        "tau":     tau,
        "n_seeds": len(runs),
        "mAP_mean":          float(mAPs.mean()),
        "mAP_std":           float(mAPs.std()),
        "f1_macro_mean":     float(f1ms.mean()),
        "f1_macro_std":      float(f1ms.std()),
        "f1_per_class_mean": f1pcs.mean(0).tolist(),
        "f1_per_class_std":  f1pcs.std(0).tolist(),
    }

out_path = CONFIG["results_dir"] / "tau_ablation.json"
with open(out_path, "w") as f:
    json.dump(agg, f, indent=2)
print(f"Saved: {out_path}")


def best_in_family(fusion, tau):
    keys = [k for k, v in agg.items()
            if v["fusion"] == fusion and v["tau"] == tau and v["caption"] != "none"]
    if not keys:
        keys = [k for k, v in agg.items() if v["fusion"] == fusion and v["tau"] == tau]
    return max(keys, key=lambda k: agg[k]["mAP_mean"])


print("\n=== Best mAP per fusion family, per tau (3-seed mean +/- std) ===")
print(f"{'fusion':12s}  " + "  ".join(f"tau={t:>2}%        " for t in CONFIG["taus"]))
for fusion in CONFIG["fusions"]:
    cells = []
    for tau in CONFIG["taus"]:
        k = best_in_family(fusion, tau)
        v = agg[k]
        cells.append(f"{v['mAP_mean']:.3f}+/-{v['mAP_std']:.3f}")
    print(f"{fusion:12s}  " + "  ".join(f"{c:>14s}" for c in cells))


print("\n=== Tau sensitivity: best-CA mAP vs image-only mAP, per tau ===")
for tau in CONFIG["taus"]:
    img = agg[f"image_only__none__tau{tau}"]
    ca  = agg[best_in_family("cross_attn", tau)]
    delta = ca["mAP_mean"] - img["mAP_mean"]
    print(f"  tau={tau:>2}%   image-only mAP {img['mAP_mean']:.3f}+/-{img['mAP_std']:.3f}"
          f"   best-CA mAP {ca['mAP_mean']:.3f}+/-{ca['mAP_std']:.3f}   delta={delta:+.3f}"
          f"   ({ca['caption']})")


Saved: /content/drive/MyDrive/Colab Notebooks/DI725/phase3_results/tau_ablation.json

=== Best mAP per fusion family, per tau (3-seed mean +/- std) ===
fusion        tau= 5%          tau=10%          tau=20%        
image_only     0.841+/-0.000   0.817+/-0.001   0.791+/-0.001
late           0.915+/-0.002   0.908+/-0.003   0.896+/-0.001
film           0.944+/-0.000   0.943+/-0.001   0.930+/-0.003
gated          0.909+/-0.002   0.897+/-0.001   0.879+/-0.009
cross_attn     0.938+/-0.001   0.948+/-0.003   0.945+/-0.002

=== Tau sensitivity: best-CA mAP vs image-only mAP, per tau ===
  tau= 5%   image-only mAP 0.841+/-0.000   best-CA mAP 0.938+/-0.001   delta=+0.097   (hybrid_gemma3-4b)
  tau=10%   image-only mAP 0.817+/-0.001   best-CA mAP 0.948+/-0.003   delta=+0.131   (hybrid_qwen3-vl-8b)
  tau=20%   image-only mAP 0.791+/-0.001   best-CA mAP 0.945+/-0.002   delta=+0.153   (hybrid_qwen3-vl-8b)


## 02 Seg Full Matrix

Full Path B (segmentation) matrix at B/32
Phase-2 ran 7 segmentation conditions (3 captions x 2 fusions + image-only).
Phase-3 closes the matrix: 5 captions x 4 fusions + image-only = 21 conds
x 3 seeds = 63 runs.

CA direction for segmentation is INVERTED from classification: image patches
are queries over text tokens (each patch asks 'what does the caption say
about my region?'). Phase-2 finding to verify at full scale: CA noise
robustness signature carries over (Late+vision_qwen ~= image-only; CA
recovers a clear delta).


In [ ]:
# 1 - Build / load 7x7 patch labels from masks
import json
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import confusion_matrix
from tqdm.auto import tqdm

assert "CONFIG" in globals(), "Run 01_tau_ablation CELL 1-2 first (CONFIG / features / text encodings)."
assert "image_patches_gpu" in globals(), "image_patches_gpu missing - re-run 01 CELL 1."
assert "text_tokens" in globals() and "text_pooled" in globals(), "text encodings missing - re-run 01 CELL 2."

MASKS_DIR = CONFIG["data_root"] / "masks"
IMG_SIZE  = 224
PATCH_B32 = 32   # 7x7 grid at 224 input
GRID_B32  = IMG_SIZE // PATCH_B32   # 7
N_PATCHES_B32 = GRID_B32 * GRID_B32  # 49

CLASS_COLORS = np.array([
    [0,   100, 0],     # 0 Tree
    [255, 182, 193],   # 1 Shrub
    [154, 205, 50],    # 2 Grass
    [255, 215, 0],     # 3 Crop
    [139, 69,  19],    # 4 Built-up
    [211, 211, 211],   # 5 Barren
    [0,   0,   255],   # 6 Water
], dtype=np.int32)

PATCH_LABELS_PATH = CONFIG["feat_dir"] / "patch_labels_7x7.pt"

if PATCH_LABELS_PATH.exists():
    cache = torch.load(PATCH_LABELS_PATH, map_location="cpu")
    patch_labels_seg = cache["patch_labels"]
    print(f"Loaded cached patch labels: {tuple(patch_labels_seg.shape)}")
else:
    def rgb_to_class_idx(rgb_mask):
        H, W, _ = rgb_mask.shape
        out = np.full((H, W), -1, dtype=np.int64)
        for idx, color in enumerate(CLASS_COLORS):
            m = ((rgb_mask[..., 0] == color[0]) &
                 (rgb_mask[..., 1] == color[1]) &
                 (rgb_mask[..., 2] == color[2]))
            out[m] = idx
        return out

    def majority_per_patch(cls_mask, patch):
        H, W = cls_mask.shape
        nh, nw = H // patch, W // patch
        out = np.zeros(nh * nw, dtype=np.int64)
        i = 0
        for h in range(nh):
            for w in range(nw):
                pat = cls_mask[h*patch:(h+1)*patch, w*patch:(w+1)*patch].flatten()
                valid = pat[pat >= 0]
                out[i] = 0 if len(valid) == 0 else int(np.bincount(valid, minlength=7).argmax())
                i += 1
        return out

    all_patches = np.zeros((len(df), N_PATCHES_B32), dtype=np.int64)
    for i, name in enumerate(tqdm(df["filename"].tolist(), desc="masks->patches")):
        rgb = np.array(Image.open(MASKS_DIR / name).resize((IMG_SIZE, IMG_SIZE), Image.NEAREST))
        cls = rgb_to_class_idx(rgb)
        all_patches[i] = majority_per_patch(cls, PATCH_B32)

    patch_labels_seg = torch.tensor(all_patches, dtype=torch.long)
    torch.save(
        {"patch_labels": patch_labels_seg, "filenames": df["filename"].tolist(),
         "patch_grid": (GRID_B32, GRID_B32), "patch_size": PATCH_B32},
        PATCH_LABELS_PATH,
    )
    print(f"Saved: {PATCH_LABELS_PATH}")

patch_labels_seg_gpu = patch_labels_seg.to(DEVICE)



Loaded cached patch labels: (10000, 49)


In [ ]:
#  2 - Segmentation modules (dim-parameterised, same as Path A modules)
DIM = CONFIG["feature_dim"]   # default 512 (B/32); override via dim=... for L/14
H   = CONFIG["hidden"]
DR  = CONFIG["dropout"]
NH  = CONFIG["n_heads"]
N_CLASSES = len(CLASSES)


class ImageOnlySeg(nn.Module):
    def __init__(self, dim=DIM, hidden=H, dropout=DR):
        super().__init__()
        self.head = nn.Sequential(
            nn.Linear(dim, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, N_CLASSES),
        )
    def forward(self, image_patches):
        return self.head(image_patches)


class LateSeg(nn.Module):
    """Broadcast pooled text to every patch, concat, classify per patch."""
    def __init__(self, dim=DIM, hidden=H, dropout=DR):
        super().__init__()
        self.head = nn.Sequential(
            nn.Linear(dim * 2, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, N_CLASSES),
        )
    def forward(self, image_patches, text_pooled_in):
        text_b = text_pooled_in.unsqueeze(1).expand(-1, image_patches.shape[1], -1)
        return self.head(torch.cat([image_patches, text_b], dim=-1))


class FiLMSeg(nn.Module):
    """Pooled text -> (gamma, beta) modulates each image patch identically.
    Zero-init keeps the model at the image-only baseline at step 0."""
    def __init__(self, dim=DIM, hidden=H, dropout=DR):
        super().__init__()
        self.modulator = nn.Sequential(
            nn.Linear(dim, hidden), nn.ReLU(),
            nn.Linear(hidden, dim * 2),
        )
        nn.init.zeros_(self.modulator[-1].weight)
        nn.init.zeros_(self.modulator[-1].bias)
        self.head = nn.Sequential(
            nn.Linear(dim, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, N_CLASSES),
        )
    def forward(self, image_patches, text_pooled_in):
        gamma, beta = self.modulator(text_pooled_in).chunk(2, dim=-1)
        gamma = gamma.unsqueeze(1)   # [B,1,dim]
        beta  = beta.unsqueeze(1)
        return self.head((1.0 + gamma) * image_patches + beta)


class GatedSeg(nn.Module):
    """Per-patch sigmoid gate over [patch || text]; mixes patch with broadcast text."""
    def __init__(self, dim=DIM, hidden=H, dropout=DR):
        super().__init__()
        self.gate = nn.Sequential(nn.Linear(dim * 2, dim), nn.Sigmoid())
        self.head = nn.Sequential(
            nn.Linear(dim, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, N_CLASSES),
        )
    def forward(self, image_patches, text_pooled_in):
        text_b = text_pooled_in.unsqueeze(1).expand(-1, image_patches.shape[1], -1)
        g = self.gate(torch.cat([image_patches, text_b], dim=-1))
        return self.head(g * image_patches + (1.0 - g) * text_b)


class CASeg(nn.Module):
    """Image patches (Q) attend over text tokens (K, V). Per-patch logits."""
    def __init__(self, dim=DIM, n_heads=NH, hidden=H, dropout=DR):
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, n_heads, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Sequential(
            nn.Linear(dim, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, N_CLASSES),
        )
    def forward(self, image_patches, text_tokens_in):
        attended, _ = self.attn(query=image_patches, key=text_tokens_in, value=text_tokens_in)
        return self.head(self.norm(attended + image_patches))


SEG_MODULES = {
    "image_only": ImageOnlySeg,
    "late":       LateSeg,
    "film":       FiLMSeg,
    "gated":      GatedSeg,
    "cross_attn": CASeg,
}


def seg_features_for(condition, caption_col):
    if condition == "image_only":
        return (image_patches_gpu.float(),)
    if condition in ("late", "film", "gated"):
        return (image_patches_gpu.float(), text_pooled[caption_col].to(DEVICE).float())
    if condition == "cross_attn":
        return (image_patches_gpu.float(), text_tokens[caption_col].to(DEVICE).float())
    raise ValueError(condition)



In [ ]:
# 3 - Training helper for segmentation (CE loss on per-patch logits)
import wandb

SEG_WANDB_PROJECT = "di725-phase3-seg-full"
SEG_BATCH_SIZE    = 128   # patch-level CE is heavier than BCE; halve batch


def compute_iou(preds_flat, target_flat, n_classes=N_CLASSES):
    ious = np.zeros(n_classes)
    for c in range(n_classes):
        pc = (preds_flat == c)
        tc = (target_flat == c)
        union = (pc | tc).sum()
        ious[c] = (pc & tc).sum() / union if union > 0 else float("nan")
    return ious, float(np.nanmean(ious))


def train_seg_one(condition, caption_col, seed):
    set_seeds(seed)
    name = f"seg_{condition}__{caption_col or 'none'}__s{seed}"

    feats   = seg_features_for(condition, caption_col)
    net     = SEG_MODULES[condition]().to(DEVICE)
    opt     = torch.optim.AdamW(net.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
    loss_fn = nn.CrossEntropyLoss()

    yt    = patch_labels_seg_gpu[train_idx]                       # [Ntr, 49]
    yv_np = patch_labels_seg_gpu[val_idx].cpu().numpy().reshape(-1)

    wandb.init(
        project=SEG_WANDB_PROJECT, name=name, reinit=True,
        tags=[f"fusion={condition}", f"caption={caption_col or 'none'}",
              f"seed={seed}", "backbone=vitb32"],
        config={"condition": condition, "caption": caption_col, "seed": seed,
                "epochs": CONFIG["epochs"], "lr": CONFIG["lr"],
                "wd": CONFIG["weight_decay"], "batch_size": SEG_BATCH_SIZE,
                "backbone": "RemoteCLIP-ViT-B/32", "patch_grid": GRID_B32},
    )

    best = {"val_mIoU": 0.0, "val_iou_per_class": None, "epoch": -1}
    tr_idx_t = torch.tensor(train_idx, device=DEVICE)
    v_idx_t  = torch.tensor(val_idx,  device=DEVICE)
    v_inputs = tuple(f[v_idx_t] for f in feats)

    for ep in range(CONFIG["epochs"]):
        net.train()
        perm = torch.randperm(len(train_idx), device=DEVICE)
        epoch_loss = 0.0; nbatch = 0
        for i in range(0, len(train_idx), SEG_BATCH_SIZE):
            b = perm[i : i + SEG_BATCH_SIZE]
            inputs = tuple(f[tr_idx_t[b]] for f in feats)
            opt.zero_grad()
            logits = net(*inputs)                                    # [B, 49, 7]
            loss = loss_fn(logits.permute(0, 2, 1), yt[b])           # [B, 7, 49]
            loss.backward(); opt.step()
            epoch_loss += loss.item(); nbatch += 1

        net.eval()
        with torch.no_grad():
            logits = net(*v_inputs)
            preds  = logits.argmax(dim=-1).cpu().numpy().reshape(-1)
        ious, miou = compute_iou(preds, yv_np)

        log = {"epoch": ep, "train/loss": epoch_loss / nbatch, "val/mIoU": miou}
        for c, v in zip(CLASSES, ious):
            log[f"val/IoU_{c}"] = float(v)
        wandb.log(log)

        if miou > best["val_mIoU"]:
            best = {"val_mIoU": miou,
                    "val_iou_per_class": [float(x) for x in ious],
                    "epoch": ep}

    wandb.finish()
    return best



In [ ]:
# 4 - Run full B/32 seg matrix: 21 conditions x 3 seeds = 63 runs
seg_results = {}   # key = (fusion, caption_or_none) -> list of best dicts
total = 0
for seed in CONFIG["seeds"]:
    print(f"\n##### SEG SEED {seed} (B/32) #####")
    key = ("image_only", "none")
    seg_results.setdefault(key, []).append(train_seg_one("image_only", None, seed))
    total += 1
    for cap in CONFIG["captions"]:
        for fusion in ("late", "film", "gated", "cross_attn"):
            key = (fusion, cap)
            seg_results.setdefault(key, []).append(train_seg_one(fusion, cap, seed))
            total += 1

print(f"\nTotal seg runs (B/32): {total}")




##### SEG SEED 42 (B/32) #####


epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train/loss,█▄▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/IoU_Barren,▁▁▁▁▄▅▆▆▇▆▇▇▇▇▇▇██▇████▇█████▇
val/IoU_Built-up,▁▁▁▁▁▃▅▆▆▇▇▇▇▇▇▇██████████████
val/IoU_Crop,▁▆▇▇▇▇▇███████████████████████
val/IoU_Grass,▁▄▅▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇██▇████████
val/IoU_Shrub,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▁▂▂▃▂▃▄▇█▄▇
val/IoU_Tree,▁▃▅▆▆▇▇▇▇▇▇▇▇▇▇▇▇███▇█████████
val/IoU_Water,▁▂▆▇▇█████████████████████████
val/mIoU,▁▃▄▅▅▆▇▇▇▇▇▇▇▇▇███████████████
epoch,29


epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train/loss,█▄▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/IoU_Barren,▁▁▄▆▆▇▇▇▇█████████████████████
val/IoU_Built-up,▁▁▁▁▄▅▇▇▇▇▇▇▇▇████████████████
val/IoU_Crop,▁▃▄▅▆▆▆▆▇▇▇▇▇▇▇▇▇▇████████████
val/IoU_Grass,▁▄▅▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇████████████
val/IoU_Shrub,▁▁▁▁▁▁▁▁▁▁▂▂▃▃▃▄▅▄▆▄▆▆▆▇▆▇█▇▆█
val/IoU_Tree,▁▄▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇█▇██████████
val/IoU_Water,▁▄▇▇▇▇▇███████████████████████
val/mIoU,▁▂▄▅▆▆▆▇▇▇▇▇▇▇▇▇▇▇█▇██████████
epoch,29


epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train/loss,█▄▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/IoU_Barren,▁▂▇▇▇▇▇██▇▇█▇█▇███████████████
val/IoU_Built-up,▁▁▆▆▇▇▇██▇████▇███████████████
val/IoU_Crop,▁▃▅▄▆▆▆▇▇▇▇▇▇▇▇▇▆█▇███▇███████
val/IoU_Grass,▁▄▅▆▆▇▆▇▇▇▇▆▇█▇█▇████▇████████
val/IoU_Shrub,▁▁▁▁▃▂▅▆▆▅▆▅▆▆▄█▆▆▅▇▆▆▅█▇▆█▇██
val/IoU_Tree,▁▅▆▆▆▆▆▇▇▇▇▇▇▇████████▇███████
val/IoU_Water,▁▆▇▇██████████████████████████
val/mIoU,▁▃▆▆▇▇▇▇▇▇▇▇▇█▇███████████████
epoch,29


epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train/loss,█▄▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/IoU_Barren,▁▁▁▄▆▆▇▇▇▇▇██▇████████████████
val/IoU_Built-up,▁▁▁▁▁▂▄▅▆▇▇▇▇▇▇▇██████████████
val/IoU_Crop,▁▇▇▇▇▇▇███████████████████████
val/IoU_Grass,▁▄▅▆▇▇▇▇▇▇▇▇▇█▇███████████████
val/IoU_Shrub,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▃▄▄▅▄▅▆▆▆▅▇██▆█
val/IoU_Tree,▁▃▅▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇█████████
val/IoU_Water,▁▁▅▇▇▇▇▇▇█████████████████████
val/mIoU,▁▂▃▅▅▅▆▆▇▇▇▇▇▇▇▇▇▇█▇██████████
epoch,29


...

epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train/loss,█▄▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/IoU_Barren,▁▁▁▂▃▅▆▆▆▆▇▇▇▇▇▇█▇█▇▇███▇▇████
val/IoU_Built-up,▁▁▁▂▅▆▇▇▇▇▇███████████████████
val/IoU_Crop,▁▄▅▅▆▆▇▇▆▇▇▇▇▇▇██████████▇████
val/IoU_Grass,▁▃▄▅▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇█▇▇▇▇████▇█
val/IoU_Shrub,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▁▂▂▂▃▃▃▇▄▂▆▆▇█
val/IoU_Tree,▁▃▅▆▆▇▇▇▇▇▇▇▇▇▇████▇██████████
val/IoU_Water,▁▃▇▇▇█████████████████████████
val/mIoU,▁▂▄▅▆▆▇▇▇▇▇▇▇▇▇███████████████
epoch,29


epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train/loss,█▅▄▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/IoU_Barren,▁▁▅▆▁▅▆▆▇▇▇▆▇▇██▇████▇▇█▇▇██▇█
val/IoU_Built-up,▁▁▅▇▅▇▇▇███▇█████████▇████████
val/IoU_Crop,▁▃▅▅▅▆▆▆▆▇▇▇███▇▇███▇▇████████
val/IoU_Grass,▁▄▅▆▅▆▆▇▇▇▇▇█▇███████████▇████
val/IoU_Shrub,▁▁▁▁▁▂▄▅▅▅▆▆▅▅▄██▇▆▅▅▄▇▇▆▆▇▆▆▇
val/IoU_Tree,▁▃▅▆▅▆▇▇▇▇▇████████████▇██████
val/IoU_Water,▁▆▇▇▇█████████████████████████
val/mIoU,▁▃▆▆▅▇▇▇▇██▇█████████▇████████
epoch,29


epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train/loss,█▄▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/IoU_Barren,▁▁▁▁▁▃▆▆▆▆▇▇▇▇▇▇▇▇█▇▇▇▇█▇▇▇███
val/IoU_Built-up,▁▁▁▁▁▂▄▅▆▇▇▇▇█████████████████
val/IoU_Crop,▁▄▅▅▆▆▇▇▇▇▇▇▇▇▇▇▇█████████████
val/IoU_Grass,▁▃▄▅▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇███████████
val/IoU_Shrub,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▅▂▂▄▄▇█
val/IoU_Tree,▁▃▅▆▆▇▇▇▇▇▇▇▇▇████████████████
val/IoU_Water,▁▁▅▇▇▇▇▇██████████████████████
val/mIoU,▁▂▃▄▄▅▆▆▇▇▇▇▇▇▇▇██████████████
epoch,29


epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train/loss,█▅▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁
val/IoU_Barren,▁▄▄▅▇▄▇▅▅█▇▇▅▇▅▆██▆▅▅▆█▆▆█▇▆▇▆
val/IoU_Built-up,▁▆▆▇▇▇▇▇▇▇▇████▇█▇▇▇▇▇█▇▇▇▇█▇█
val/IoU_Crop,▁▃▅▆▇▆▇▇█▇▆▇█▇▇███▇▇▆▇▇▇▇▇▄▇▇▇
val/IoU_Grass,▁▂▄▅▆▆▆▇███▆▇█▇▇▇██▇█▇▇█▅▇▇▇▇▇
val/IoU_Shrub,▁▂▄▄▅▆▇▃▆▅▄▅▃█▇█▇█▇▅▄▂▄▇▇▇▇▅▇▅
val/IoU_Tree,▁▁▄▆▇▇▇▇▇█▇█▇█████▇█████▇█▇▆█▇
val/IoU_Water,▁▄▃▄▇▆▇▇▇▇▇▇▇▆▇███▇▇▇▇▇▇█▇▇▇▇▇
val/mIoU,▁▄▅▆▇▆▇▆▇█▇▇▇█▇███▇▇▇▇█▇▇█▇▇▇▇
epoch,29



Total seg runs (B/32): 63


In [ ]:
# 5 - Aggregate, save JSON, print headline tables
seg_agg = {}
for (fusion, cap), runs in seg_results.items():
    miou = np.array([r["val_mIoU"] for r in runs])
    iouc = np.array([r["val_iou_per_class"] for r in runs])
    seg_agg[f"{fusion}__{cap}"] = {
        "fusion": fusion, "caption": cap, "n_seeds": len(runs),
        "mIoU_mean": float(miou.mean()), "mIoU_std": float(miou.std()),
        "iou_per_class_mean": iouc.mean(0).tolist(),
        "iou_per_class_std":  iouc.std(0).tolist(),
    }

out_path = CONFIG["results_dir"] / "seg_full_b32.json"
with open(out_path, "w") as f:
    json.dump(seg_agg, f, indent=2)
print(f"Saved: {out_path}")

print("\n=== mIoU heatmap: fusion (rows) x caption (cols), B/32 ===")
img_a = seg_agg["image_only__none"]
print(f"  image-only baseline: {img_a['mIoU_mean']:.3f}+/-{img_a['mIoU_std']:.3f}")
print(f"{'fusion':12s}  " + "  ".join(f"{c[:14]:>14s}" for c in CONFIG["captions"]) + "       mean")
for fusion in ("late", "film", "gated", "cross_attn"):
    row = [seg_agg[f"{fusion}__{cap}"]["mIoU_mean"] for cap in CONFIG["captions"]]
    mean_r = sum(row) / len(row)
    print(f"{fusion:12s}  " + "  ".join(f"{m:>14.3f}" for m in row) + f"  {mean_r:.3f}")

print("\n=== CA vs Late delta per caption (B/32, full matrix) ===")
for cap in CONFIG["captions"]:
    late_v = seg_agg[f"late__{cap}"]
    ca_v   = seg_agg[f"cross_attn__{cap}"]
    delta  = ca_v["mIoU_mean"] - late_v["mIoU_mean"]
    print(f"  {cap:25s}  late {late_v['mIoU_mean']:.3f}+/-{late_v['mIoU_std']:.3f}"
          f"  ->  CA {ca_v['mIoU_mean']:.3f}+/-{ca_v['mIoU_std']:.3f}   delta={delta:+.3f}")

print("\n=== Per-class IoU lift: best CA vs image-only (B/32) ===")
best_ca_key = max((k for k, v in seg_agg.items() if v["fusion"] == "cross_attn"),
                  key=lambda k: seg_agg[k]["mIoU_mean"])
best_ca = seg_agg[best_ca_key]
print(f"  Best CA: {best_ca_key}  mIoU {best_ca['mIoU_mean']:.3f}+/-{best_ca['mIoU_std']:.3f}")
for c, base_m, ca_m in zip(CLASSES, img_a["iou_per_class_mean"], best_ca["iou_per_class_mean"]):
    print(f"    {c:10s}  image-only {base_m:.3f}  ->  best-CA {ca_m:.3f}   delta={ca_m - base_m:+.3f}")


Saved: /content/drive/MyDrive/Colab Notebooks/DI725/phase3_results/seg_full_b32.json

=== mIoU heatmap: fusion (rows) x caption (cols), B/32 ===
  image-only baseline: 0.574+/-0.003
fusion        hybrid_gemma3-  hybrid_qwen3-v   text_qwen3-4b  vision_gemma3-  vision_qwen3-v       mean
late                   0.663           0.669           0.673           0.582           0.575  0.633
film                   0.689           0.694           0.693           0.615           0.613  0.661
gated                  0.660           0.667           0.675           0.581           0.573  0.631
cross_attn             0.709           0.711           0.701           0.608           0.610  0.668

=== CA vs Late delta per caption (B/32, full matrix) ===
  hybrid_gemma3-4b           late 0.663+/-0.001  ->  CA 0.709+/-0.002   delta=+0.046
  hybrid_qwen3-vl-8b         late 0.669+/-0.001  ->  CA 0.711+/-0.003   delta=+0.042
  text_qwen3-4b              late 0.673+/-0.002  ->  CA 0.701+/-0.002   delta=+0.028
 

## 03A Extract L14 Features

RemoteCLIP-ViT-L/14 patch + text features
Re-encode the dataset with the L/14 backbone for the headline 'severely
affects results' ablation. The B/16 checkpoint does not exist on HF
(chendelong/RemoteCLIP only ships RN50, ViT-B/32, ViT-L/14), so we
upgrade in two dimensions at once:
  - patch grid: 7x7 (49 patches) -> 16x16 (256 patches)           [+5.2x spatial]
  - backbone capacity: ViT-B/32 (86M) -> ViT-L/14 (304M)          [+3.5x params]
  - feature dim: 512 -> 768                                       [output embed]

In [ ]:
# 1 - Load RemoteCLIP-ViT-L/14
import torch
from huggingface_hub import hf_hub_download
import open_clip

assert "CONFIG" in globals(), "Run 01_tau_ablation CELL 1 first."

L14_CKPT_FILE = "RemoteCLIP-ViT-L-14.pt"
L14_MODEL_NAME = "ViT-L-14"
L14_PATCH = 14
L14_GRID  = 224 // L14_PATCH              # 16
L14_N_PATCHES = L14_GRID * L14_GRID       # 256
L14_N_TOKENS  = 1 + L14_N_PATCHES         # 257
L14_FEAT_DIM  = 768

if "model_l14" not in globals():
    print(f"Loading RemoteCLIP-{L14_MODEL_NAME}...")
    ckpt_path_l14 = hf_hub_download("chendelong/RemoteCLIP", L14_CKPT_FILE)
    model_l14, _, preprocess_l14 = open_clip.create_model_and_transforms(L14_MODEL_NAME)
    msg = model_l14.load_state_dict(torch.load(ckpt_path_l14, map_location="cpu"))
    print("Load message:", msg)
    model_l14 = model_l14.to(DEVICE).eval()
    tokenizer_l14 = open_clip.get_tokenizer(L14_MODEL_NAME)
else:
    print("model_l14 already loaded.")

with torch.no_grad():
    dummy = torch.randn(2, 3, 224, 224, device=DEVICE)
    out = model_l14.encode_image(dummy)
    print(f"Pooled image out: {tuple(out.shape)}  (expected [2, {L14_FEAT_DIM}])")
    assert out.shape[-1] == L14_FEAT_DIM, f"L/14 output dim mismatch: {out.shape[-1]}"



Loading RemoteCLIP-ViT-L-14...


RemoteCLIP-ViT-L-14.pt:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

Load message: <All keys matched successfully>
Pooled image out: (2, 768)  (expected [2, 768])


In [ ]:
# 2 - Hook-based patch encoder for L/14
@torch.no_grad()
def encode_image_patches_l14(images, normalize=True):
    """[B, 3, 224, 224] -> [B, 257, 768]. Token 0 = CLS, 1..256 = 16x16 patches."""
    visual = model_l14.visual
    captured = {}

    def hook(_, __, out):
        captured["tokens"] = out          # [B, 257, vision_width]

    h = visual.ln_post.register_forward_hook(hook)
    try:
        _ = model_l14.encode_image(images)
    finally:
        h.remove()

    tokens = captured["tokens"]
    if visual.proj is not None:
        tokens = tokens @ visual.proj      # [B, 257, 768]
    if normalize:
        tokens = tokens / tokens.norm(dim=-1, keepdim=True)
    return tokens


with torch.no_grad():
    dummy = torch.randn(2, 3, 224, 224, device=DEVICE)
    patch_out = encode_image_patches_l14(dummy)
print(f"Patch out: {tuple(patch_out.shape)}  (expected [2, {L14_N_TOKENS}, {L14_FEAT_DIM}])")
assert patch_out.shape == (2, L14_N_TOKENS, L14_FEAT_DIM), "L/14 patch shape unexpected"



Patch out: (2, 257, 768)  (expected [2, 257, 768])


In [ ]:
# 3 - Encode all 10K images at L/14 patch level (or load from cache)
from pathlib import Path

from PIL import Image
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

L14_FEAT_PATH = CONFIG["feat_dir"] / "patch_features_remoteclip_vitl14.pt"

if L14_FEAT_PATH.exists():
    print(f"L/14 features already cached: {L14_FEAT_PATH}")
    cache = torch.load(L14_FEAT_PATH, map_location="cpu")
    patch_features_l14 = cache["patch_features"]
    assert cache["filenames"] == df["filename"].tolist(), "filename order drift"
    print(f"Shape: {tuple(patch_features_l14.shape)}  dtype={patch_features_l14.dtype}")
else:
    class ImgDS(Dataset):
        def __init__(self, names, transform):
            self.names = names; self.t = transform
        def __len__(self): return len(self.names)
        def __getitem__(self, i):
            img = Image.open(CONFIG["data_root"] / "images" / self.names[i]).convert("RGB")
            return self.t(img)

    ds = ImgDS(df["filename"].tolist(), preprocess_l14)
    loader = DataLoader(ds, batch_size=64, num_workers=4, shuffle=False, pin_memory=True)

    chunks = []
    with torch.no_grad():
        for x in tqdm(loader, desc="patch-encode L/14"):
            x = x.to(DEVICE, non_blocking=True)
            f = encode_image_patches_l14(x)
            chunks.append(f.cpu().half())
    patch_features_l14 = torch.cat(chunks, dim=0)
    print(f"Patch features (L/14): {tuple(patch_features_l14.shape)}  "
          f"{patch_features_l14.numel() * 2 / 1e9:.2f} GB FP16")

    torch.save(
        {"patch_features": patch_features_l14,
         "filenames": df["filename"].tolist(),
         "model": f"RemoteCLIP-{L14_MODEL_NAME}",
         "patch_grid": (L14_GRID, L14_GRID),
         "patch_size": L14_PATCH,
         "feature_dim": L14_FEAT_DIM,
         "shape_note": f"[N, 1+{L14_N_PATCHES} tokens, {L14_FEAT_DIM}]; token 0 = CLS, 1..{L14_N_PATCHES} = {L14_GRID}x{L14_GRID} patches row-major"},
        L14_FEAT_PATH,
    )
    print(f"Saved: {L14_FEAT_PATH}")

patch_features_l14_gpu = patch_features_l14.to(DEVICE)
image_cls_l14_gpu     = patch_features_l14_gpu[:, 0, :]                # [10000, 768]
image_patches_l14_gpu = patch_features_l14_gpu[:, 1:, :]               # [10000, 256, 768]
print(f"image_cls_l14={tuple(image_cls_l14_gpu.shape)}  "
      f"image_patches_l14={tuple(image_patches_l14_gpu.shape)}")



patch-encode L/14:   0%|          | 0/157 [00:00<?, ?it/s]

Patch features (L/14): (10000, 257, 768)  3.95 GB FP16
Saved: /content/drive/MyDrive/Colab Notebooks/DI725/phase2_features/patch_features_remoteclip_vitl14.pt
image_cls_l14=(10000, 768)  image_patches_l14=(10000, 256, 768)


In [ ]:
# 4 - Text encoding with L/14 (re-encode because L/14 has its own text weights)
@torch.no_grad()
def encode_text_tokens_and_pooled_l14(captions, batch_size=256):
    all_tokens, all_pooled = [], []
    for i in range(0, len(captions), batch_size):
        batch = captions[i : i + batch_size]
        ids = tokenizer_l14(batch).to(DEVICE)

        captured = {}
        def hook(_, __, out):
            captured["tokens"] = out

        h = model_l14.ln_final.register_forward_hook(hook)
        try:
            pooled = model_l14.encode_text(ids)
        finally:
            h.remove()

        tokens = captured["tokens"] @ model_l14.text_projection
        tokens = tokens / tokens.norm(dim=-1, keepdim=True)
        pooled = pooled / pooled.norm(dim=-1, keepdim=True)
        all_tokens.append(tokens.cpu().half())
        all_pooled.append(pooled.cpu().half())
    return torch.cat(all_tokens), torch.cat(all_pooled)


text_tokens_l14 = {}
text_pooled_l14 = {}
for col in CONFIG["captions"]:
    print(f"Encoding {col} with L/14...")
    captions = df[col].fillna("").tolist()
    tk, pl = encode_text_tokens_and_pooled_l14(captions)
    text_tokens_l14[col] = tk
    text_pooled_l14[col] = pl
    print(f"  tokens={tuple(tk.shape)} pooled={tuple(pl.shape)}")


Encoding hybrid_gemma3-4b with L/14...
  tokens=(10000, 77, 768) pooled=(10000, 768)
Encoding hybrid_qwen3-vl-8b with L/14...
  tokens=(10000, 77, 768) pooled=(10000, 768)
Encoding text_qwen3-4b with L/14...
  tokens=(10000, 77, 768) pooled=(10000, 768)
Encoding vision_gemma3-4b with L/14...
  tokens=(10000, 77, 768) pooled=(10000, 768)
Encoding vision_qwen3-vl-8b with L/14...
  tokens=(10000, 77, 768) pooled=(10000, 768)


## 03B Patha L14

Path A multi-label classification on L/14 features
Re-run the Path A sweep with L/14 image+text features to check whether the
fusion ranking (CA > FiLM > Late > Gated) and CA dominance still hold under
a larger, higher-resolution backbone. Path A uses pooled image CLS so the
patch-resolution gain mostly benefits CA (which attends over patches).

Conditions: 1 image-only + 5 captions x 4 fusions = 21 conditions
x 3 seeds = 63 runs.



In [ ]:
# %% CELL 1 - Sanity assertions
import json
import numpy as np
import torch
import torch.nn as nn
import wandb
from sklearn.metrics import average_precision_score, f1_score

for g in ("CONFIG", "train_idx", "val_idx", "set_seeds",
          "ImageOnlyHead", "LateFusion", "FiLMFusion", "GatedFusion", "CrossAttentionFusion",
          "image_cls_l14_gpu", "image_patches_l14_gpu",
          "text_tokens_l14", "text_pooled_l14",
          "labels_by_tau", "L14_FEAT_DIM"):
    assert g in globals(), f"{g} missing - run upstream cells (01, 03a) first."



In [ ]:
# %% CELL 2 - Feature selectors and module factory for L/14 Path A
def features_for_l14(condition, caption_col):
    if condition == "image_only":
        return (image_cls_l14_gpu.float(),)
    if condition in ("late", "film", "gated"):
        return (image_cls_l14_gpu.float(), text_pooled_l14[caption_col].to(DEVICE).float())
    if condition == "cross_attn":
        return (image_patches_l14_gpu.float(), text_tokens_l14[caption_col].to(DEVICE).float())
    raise ValueError(condition)


L14_MODULE_CLS = {
    "image_only": ImageOnlyHead,
    "late":       LateFusion,
    "film":       FiLMFusion,
    "gated":      GatedFusion,
    "cross_attn": CrossAttentionFusion,
}


def make_l14_module(condition):
    return L14_MODULE_CLS[condition](dim=L14_FEAT_DIM).to(DEVICE)


L14_TAU = 10
L14_PATHA_WANDB_PROJECT = "di725-phase3-l14-pathA"


def train_pathA_l14(condition, caption_col, seed):
    set_seeds(seed)
    name = f"l14_{condition}__{caption_col or 'none'}__s{seed}"

    feats   = features_for_l14(condition, caption_col)
    net     = make_l14_module(condition)
    opt     = torch.optim.AdamW(net.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
    loss_fn = nn.BCEWithLogitsLoss()

    y_all = labels_by_tau[L14_TAU]
    yt    = y_all[train_idx]
    yv_np = y_all[val_idx].cpu().numpy()

    wandb.init(
        project=L14_PATHA_WANDB_PROJECT, name=name, reinit=True,
        tags=[f"fusion={condition}", f"caption={caption_col or 'none'}",
              f"seed={seed}", "backbone=vitl14", f"tau={L14_TAU}"],
        config={"condition": condition, "caption": caption_col, "tau": L14_TAU,
                "seed": seed, "epochs": CONFIG["epochs"], "lr": CONFIG["lr"],
                "wd": CONFIG["weight_decay"], "batch_size": CONFIG["batch_size"],
                "backbone": "RemoteCLIP-ViT-L/14", "feature_dim": L14_FEAT_DIM},
    )

    best = {"val_mAP": 0.0, "val_f1_macro": 0.0, "val_f1_per_class": None, "epoch": -1}
    tr_idx_t = torch.tensor(train_idx, device=DEVICE)
    v_idx_t  = torch.tensor(val_idx,  device=DEVICE)
    v_inputs = tuple(f[v_idx_t] for f in feats)

    for ep in range(CONFIG["epochs"]):
        net.train()
        perm = torch.randperm(len(train_idx), device=DEVICE)
        epoch_loss = 0.0; nbatch = 0
        for i in range(0, len(train_idx), CONFIG["batch_size"]):
            b = perm[i : i + CONFIG["batch_size"]]
            inputs = tuple(f[tr_idx_t[b]] for f in feats)
            opt.zero_grad()
            loss = loss_fn(net(*inputs), yt[b])
            loss.backward(); opt.step()
            epoch_loss += loss.item(); nbatch += 1

        net.eval()
        with torch.no_grad():
            logits = net(*v_inputs).cpu().numpy()
        probs = 1.0 / (1.0 + np.exp(-logits))
        preds = (probs > 0.5).astype(int)
        mAP   = average_precision_score(yv_np, probs, average="macro")
        f1m   = f1_score(yv_np, preds, average="macro", zero_division=0)
        f1pc  = f1_score(yv_np, preds, average=None,    zero_division=0)

        log = {"epoch": ep, "train/loss": epoch_loss / nbatch,
               "val/mAP": mAP, "val/f1_macro": f1m}
        for c, f in zip(CLASSES, f1pc):
            log[f"val/f1_{c}"] = f
        wandb.log(log)

        if mAP > best["val_mAP"]:
            best = {"val_mAP": float(mAP), "val_f1_macro": float(f1m),
                    "val_f1_per_class": [float(x) for x in f1pc], "epoch": ep}

    wandb.finish()
    return best



In [ ]:
m = ImageOnlyHead(dim=768)
print(f"OK: ImageOnlyHead accepts dim parameter. First layer: {m.net[0]}")

OK: ImageOnlyHead accepts dim parameter. First layer: Linear(in_features=768, out_features=256, bias=True)


In [ ]:
# %% CELL 3 - Sweep: 1 image-only + 5 cap x 4 fusion = 21 cond x 3 seeds = 63 runs
l14_pathA_results = {}
total = 0
for seed in CONFIG["seeds"]:
    print(f"\n##### L/14 Path A SEED {seed} #####")
    l14_pathA_results.setdefault(("image_only", "none"), []).append(
        train_pathA_l14("image_only", None, seed))
    total += 1
    for cap in CONFIG["captions"]:
        for fusion in ("late", "film", "gated", "cross_attn"):
            l14_pathA_results.setdefault((fusion, cap), []).append(
                train_pathA_l14(fusion, cap, seed))
            total += 1
print(f"\nTotal L/14 Path A runs: {total}")




##### L/14 Path A SEED 42 #####


epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train/loss,█▅▄▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/f1_Barren,▁▁▁▁▁▁▁▂▃▄▄▆▆▆▆▆▇▆▇▇▇██▇██████
val/f1_Built-up,▁▁▁▁▁▁▁▁▁▁▁▁▁▂▂▂▄▅▆▆▆▇▇▇▇█████
val/f1_Crop,▁▆████████████████████████████
val/f1_Grass,▁▁▁▁▁▂▃▄▅▆▇▆▇▇▇▇▇▇▇█▇██▇██████
val/f1_Shrub,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/f1_Tree,▁▆▆▆▇▇▇▇▇▇▇▇▇█████████████████
val/f1_Water,▁▁▁▁▁▁▁▂▂▄▆▆▇▇▇███████████████
val/f1_macro,▁▃▃▃▃▃▃▄▄▅▅▆▆▆▆▆▇▇▇▇▇█████████
+1,...


epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train/loss,█▅▄▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/f1_Barren,▁▁▁▁▃▄▅▅▆▆▆▇▇▇▇▇▇▇▇▇▇██▇██████
val/f1_Built-up,▁▁▁▁▁▁▁▁▁▂▂▂▄▅▅▆▆▆▆▇▇▇██▇█████
val/f1_Crop,▁▇▇▇██████████████████████████
val/f1_Grass,▁▁▁▂▅▆▇▇▇▇▇▇▇▇▇█▇█████████████
val/f1_Shrub,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▂▂▂▂▂▅▂█▅▇
val/f1_Tree,▁▃▄▄▅▅▅▅▆▆▆▆▇▇▇▇▇▇▇▇██████████
val/f1_Water,▁▁▁▁▂▅▆▇▇▇████████████████████
val/f1_macro,▁▃▃▃▃▄▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇█████
+1,...


epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train/loss,█▅▄▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/f1_Barren,▁▁▁▃▄▅▅▆▆▇▇▇▇█▇██▇████████████
val/f1_Built-up,▁▁▁▁▁▂▄▅▆▇▇▇▆█▇▇█████████▇████
val/f1_Crop,▁▇▇▇▇█▇███████████████████████
val/f1_Grass,▁▂▃▄▅▆▅▆▇▇▇▇▇▇█▇▇█████████████
val/f1_Shrub,▁▁▁▁▁▁▁▁▂▁▄▇▅▅▄▇▇▇▇▆▇██▇▇███▇▇
val/f1_Tree,▁▁▂▃▄▄▄▆▆▇▇▇▇▇▇▇▇▇█▇▇██▇██████
val/f1_Water,▁▁▆▇▇█▇███████████████████████
val/f1_macro,▁▂▄▄▅▅▅▆▆▆▇▇▇▇▇███████████████
+1,...


epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train/loss,█▅▄▃▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/f1_Barren,▁▁▁▁▁▁▁▂▃▄▄▅▅▅▆▆▆▇▇▇▇▇▇▇▇▇████
val/f1_Built-up,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▃▃▄▅▅▆▆▆▇▇█▇█
val/f1_Crop,▁▅▇▇██████████████████████████
val/f1_Grass,▁▁▁▁▁▁▄▅▅▆▆▇▇▇▇▇▇▇▇▇▇█████████
val/f1_Shrub,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/f1_Tree,▁▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇██████████████
val/f1_Water,▁▁▁▁▁▁▁▁▂▃▆▇▇█████████████████
val/f1_macro,▁▃▃▃▃▃▃▄▄▄▅▆▆▆▆▆▆▇▇▇▇▇█▇██████
+1,...


...

epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train/loss,█▅▄▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/f1_Barren,▁▁▁▁▃▃▄▄▅▅▅▅▅▆▆▅▆▆▆▇▇▇▇▇▇█▇██▇
val/f1_Built-up,▁▁▁▁▁▁▂▃▄▄▄▅▅▆▆▆▆▆▆▇▇█▇███▇█▇█
val/f1_Crop,▁▇▇▇██████████████████████████
val/f1_Grass,▁▁▁▁▂▄▅▅▅▆▆▆▆▇▇▇▇▇▇▇▇▇▇███████
val/f1_Shrub,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█▁▁██
val/f1_Tree,▁▁▂▃▃▄▅▅▆▆▆▇▇▇▇▇▇▇▇▇▇█████████
val/f1_Water,▁▁▁▁▄▇▇▇▇▇████████████████████
val/f1_macro,▁▂▂▃▄▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇▇█▇███████
+1,...


epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train/loss,█▅▄▄▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁
val/f1_Barren,▁▁▃▂▃▄▇▆▆▇█▇▇▇█▇███▇██▇▇████▇▇
val/f1_Built-up,▁▁▁▁▁▆▆▇▇▇▇▇▇███▇▇██▇█▇▇▇▇▇▇▇█
val/f1_Crop,▁▄▅▆▆▇▇▇▇▇▇▇▇▇█▇▇██▇███▇██████
val/f1_Grass,▂▁▁▃▄▅▆▆▇▇▇▇█▇█▇███▄██▆▆█████▇
val/f1_Shrub,▁▁▁▁▁▁▁▃▁▁▁▃▁▄▁▁▇▇▃▂▆█▅▅▇█▂▇██
val/f1_Tree,▁▂▃▄▅▇▇▇▇▇████▆▇███▇████████▇█
val/f1_Water,▁▁▆▇▇▇▇███████████████████████
val/f1_macro,▁▁▃▃▄▆▆▆▆▇▇▇▇▇▇▇██▇▇██▇▇██▇███
+1,...


epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train/loss,█▅▄▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/f1_Barren,▁▁▁▁▁▁▂▂▄▅▅▅▅▆▆▆▆▇▇▇▇█▇▇▇████▇
val/f1_Built-up,▁▁▁▁▁▁▁▁▁▁▁▁▂▃▃▄▅▅▅▆▆▆▇▇▇█▇█▇█
val/f1_Crop,▁▄▇▇██████████████████████████
val/f1_Grass,▁▁▁▁▁▂▂▄▅▅▅▆▆▇▇▇▇▇▇▇▇█████████
val/f1_Shrub,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/f1_Tree,▁▁▂▃▃▄▅▅▅▆▆▇▇▇▇▇▇▇▇▇▇▇████████
val/f1_Water,▁▁▁▁▁▂▄▆▇█████████████████████
val/f1_macro,▁▂▃▃▃▃▄▅▅▆▆▆▆▆▇▇▇▇▇▇▇█████████
+1,...


epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train/loss,█▅▄▃▃▃▃▃▃▃▂▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁
val/f1_Barren,▁▅▆▅▇▇▇▄██▆▆▇▇▇▆▇█▅█▇▆███▆▇███
val/f1_Built-up,▁▆▆▇▇▇████▇▇▇█▇▇██▇██████▇▇██▇
val/f1_Crop,▁▁▅▅▆▆▇▆▄▇▇▇▇█▄▇▇▇▅█▆██▄▆▆▇▅▇▇
val/f1_Grass,▁▆▆▇▄▆▇▆▇▇▇▇▇▆▆█▆▇▇▆▇██▇▇▇▆▇█▇
val/f1_Shrub,▁▁▁▆▃▃▃▃▆▂▇▃▄▄▇█▄▇▅▇█▇▇▅▆▇▇▇█▄
val/f1_Tree,▁▂▅▆▇▄█▇▆▇▇▄▇█▇▇██▆▇▆▇█▇▆▇▅▆▆▇
val/f1_Water,▁▇▇█▇█▇██▇███▇███████████▇███▇
val/f1_macro,▁▅▅▇▆▇▇▆█▇▇▇▇▇▇▇▇█▇████▇▇▇▇██▇
+1,...



Total L/14 Path A runs: 63


In [ ]:
# %% CELL 4 - Aggregate, save, print headline (vs Phase-2/B/32 numbers)
l14_pathA_agg = {}
for (fusion, cap), runs in l14_pathA_results.items():
    mAPs  = np.array([r["val_mAP"] for r in runs])
    f1ms  = np.array([r["val_f1_macro"] for r in runs])
    f1pcs = np.array([r["val_f1_per_class"] for r in runs])
    l14_pathA_agg[f"{fusion}__{cap}"] = {
        "fusion": fusion, "caption": cap, "n_seeds": len(runs),
        "mAP_mean": float(mAPs.mean()), "mAP_std": float(mAPs.std()),
        "f1_macro_mean": float(f1ms.mean()), "f1_macro_std": float(f1ms.std()),
        "f1_per_class_mean": f1pcs.mean(0).tolist(),
        "f1_per_class_std":  f1pcs.std(0).tolist(),
        "backbone": "RemoteCLIP-ViT-L/14",
    }

out_path = CONFIG["results_dir"] / "pathA_l14.json"
with open(out_path, "w") as f:
    json.dump(l14_pathA_agg, f, indent=2)
print(f"Saved: {out_path}")

print("\n=== L/14 Path A mAP heatmap (3-seed mean) ===")
img = l14_pathA_agg["image_only__none"]
print(f"  image-only: {img['mAP_mean']:.3f}+/-{img['mAP_std']:.3f}")
print(f"{'fusion':12s}  " + "  ".join(f"{c[:14]:>14s}" for c in CONFIG["captions"]) + "       mean")
for fusion in ("late", "film", "gated", "cross_attn"):
    row = [l14_pathA_agg[f"{fusion}__{cap}"]["mAP_mean"] for cap in CONFIG["captions"]]
    print(f"{fusion:12s}  " + "  ".join(f"{m:>14.3f}" for m in row) + f"  {sum(row)/len(row):.3f}")


Saved: /content/drive/MyDrive/Colab Notebooks/DI725/phase3_results/pathA_l14.json

=== L/14 Path A mAP heatmap (3-seed mean) ===
  image-only: 0.803+/-0.002
fusion        hybrid_gemma3-  hybrid_qwen3-v   text_qwen3-4b  vision_gemma3-  vision_qwen3-v       mean
late                   0.878           0.891           0.882           0.820           0.811  0.856
film                   0.927           0.936           0.923           0.873           0.872  0.906
gated                  0.861           0.871           0.870           0.815           0.807  0.845
cross_attn             0.956           0.955           0.929           0.866           0.869  0.915


## 03C Seg L14

Full segmentation matrix on L/14 (16x16 patch grid)
The headline 'severely affects results' ablation. B/32 segmentation used a
7x7 patch grid (49 per image); L/14 gives 16x16 (256 per image), a +5.2x
jump in spatial resolution combined with +3.5x model capacity. Expect a
large mIoU lift, concentrated on classes that occupy small image regions
(Shrub, Built-up, Water).

Matrix: 1 image-only + 5 captions x 4 fusions = 21 cond x 3 seeds = 63 runs.



In [ ]:
# %% CELL 1 - 16x16 patch labels (re-pool 224x224 masks at patch=14)
import json
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import wandb
from PIL import Image
from tqdm.auto import tqdm

for g in ("CONFIG", "train_idx", "val_idx", "set_seeds",
          "ImageOnlySeg", "LateSeg", "FiLMSeg", "GatedSeg", "CASeg",
          "compute_iou",
          "image_patches_l14_gpu", "text_tokens_l14", "text_pooled_l14",
          "L14_FEAT_DIM", "L14_GRID"):
    assert g in globals(), f"{g} missing - run upstream cells (01, 02, 03a) first."

MASKS_DIR = CONFIG["data_root"] / "masks"
IMG_SIZE  = 224
L14_PATCH = 14
L14_N_PATCHES = L14_GRID * L14_GRID     # 256

CLASS_COLORS = np.array([
    [0,   100, 0],     # 0 Tree
    [255, 182, 193],   # 1 Shrub
    [154, 205, 50],    # 2 Grass
    [255, 215, 0],     # 3 Crop
    [139, 69,  19],    # 4 Built-up
    [211, 211, 211],   # 5 Barren
    [0,   0,   255],   # 6 Water
], dtype=np.int32)

PATCH_LABELS_L14_PATH = CONFIG["feat_dir"] / "patch_labels_16x16.pt"

if PATCH_LABELS_L14_PATH.exists():
    cache = torch.load(PATCH_LABELS_L14_PATH, map_location="cpu")
    patch_labels_l14 = cache["patch_labels"]
    print(f"Loaded cached 16x16 patch labels: {tuple(patch_labels_l14.shape)}")
else:
    def rgb_to_class_idx(rgb_mask):
        H, W, _ = rgb_mask.shape
        out = np.full((H, W), -1, dtype=np.int64)
        for idx, color in enumerate(CLASS_COLORS):
            m = ((rgb_mask[..., 0] == color[0]) &
                 (rgb_mask[..., 1] == color[1]) &
                 (rgb_mask[..., 2] == color[2]))
            out[m] = idx
        return out

    def majority_per_patch(cls_mask, patch):
        H, W = cls_mask.shape
        nh, nw = H // patch, W // patch
        out = np.zeros(nh * nw, dtype=np.int64)
        i = 0
        for h in range(nh):
            for w in range(nw):
                pat = cls_mask[h*patch:(h+1)*patch, w*patch:(w+1)*patch].flatten()
                valid = pat[pat >= 0]
                out[i] = 0 if len(valid) == 0 else int(np.bincount(valid, minlength=7).argmax())
                i += 1
        return out

    all_patches = np.zeros((len(df), L14_N_PATCHES), dtype=np.int64)
    for i, name in enumerate(tqdm(df["filename"].tolist(), desc="masks->16x16 patches")):
        rgb = np.array(Image.open(MASKS_DIR / name).resize((IMG_SIZE, IMG_SIZE), Image.NEAREST))
        cls = rgb_to_class_idx(rgb)
        all_patches[i] = majority_per_patch(cls, L14_PATCH)
    patch_labels_l14 = torch.tensor(all_patches, dtype=torch.long)
    torch.save(
        {"patch_labels": patch_labels_l14, "filenames": df["filename"].tolist(),
         "patch_grid": (L14_GRID, L14_GRID), "patch_size": L14_PATCH},
        PATCH_LABELS_L14_PATH,
    )
    print(f"Saved: {PATCH_LABELS_L14_PATH}")

patch_labels_l14_gpu = patch_labels_l14.to(DEVICE)

print("Class distribution at 16x16 patch level (% of 10K x 256 patches):")
for c, n in enumerate(np.bincount(patch_labels_l14.flatten().numpy(), minlength=7)):
    print(f"  {CLASSES[c]:10s}: {int(n):8d}  ({100*n/(len(df)*L14_N_PATCHES):.2f}%)")



masks->16x16 patches:   0%|          | 0/10000 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/Colab Notebooks/DI725/phase2_features/patch_labels_16x16.pt
Class distribution at 16x16 patch level (% of 10K x 256 patches):
  Tree      :   739109  (28.87%)
  Shrub     :    18480  (0.72%)
  Grass     :  1178221  (46.02%)
  Crop      :   463997  (18.12%)
  Built-up  :    26508  (1.04%)
  Barren    :    89563  (3.50%)
  Water     :    44122  (1.72%)


In [ ]:
# %% CELL 2 - Seg modules and training helper at L/14 (uses dim=L14_FEAT_DIM)
SEG_L14_WANDB_PROJECT = "di725-phase3-l14-seg"
SEG_L14_BATCH = 48   # 256 patches/image + dim=768 -> heavier; lower batch

L14_SEG_MODULE_CLS = {
    "image_only": ImageOnlySeg,
    "late":       LateSeg,
    "film":       FiLMSeg,
    "gated":      GatedSeg,
    "cross_attn": CASeg,
}


def make_l14_seg_module(condition):
    return L14_SEG_MODULE_CLS[condition](dim=L14_FEAT_DIM).to(DEVICE)


def seg_features_for_l14(condition, caption_col):
    if condition == "image_only":
        return (image_patches_l14_gpu.float(),)
    if condition in ("late", "film", "gated"):
        return (image_patches_l14_gpu.float(), text_pooled_l14[caption_col].to(DEVICE).float())
    if condition == "cross_attn":
        return (image_patches_l14_gpu.float(), text_tokens_l14[caption_col].to(DEVICE).float())
    raise ValueError(condition)


def train_seg_l14(condition, caption_col, seed):
    set_seeds(seed)
    name = f"seg_l14_{condition}__{caption_col or 'none'}__s{seed}"

    feats   = seg_features_for_l14(condition, caption_col)
    net     = make_l14_seg_module(condition)
    opt     = torch.optim.AdamW(net.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
    loss_fn = nn.CrossEntropyLoss()

    yt    = patch_labels_l14_gpu[train_idx]
    yv_np = patch_labels_l14_gpu[val_idx].cpu().numpy().reshape(-1)

    wandb.init(
        project=SEG_L14_WANDB_PROJECT, name=name, reinit=True,
        tags=[f"fusion={condition}", f"caption={caption_col or 'none'}",
              f"seed={seed}", "backbone=vitl14"],
        config={"condition": condition, "caption": caption_col, "seed": seed,
                "epochs": CONFIG["epochs"], "lr": CONFIG["lr"],
                "wd": CONFIG["weight_decay"], "batch_size": SEG_L14_BATCH,
                "backbone": "RemoteCLIP-ViT-L/14", "patch_grid": L14_GRID,
                "feature_dim": L14_FEAT_DIM},
    )

    best = {"val_mIoU": 0.0, "val_iou_per_class": None, "epoch": -1}
    tr_idx_t = torch.tensor(train_idx, device=DEVICE)
    v_idx_t  = torch.tensor(val_idx,  device=DEVICE)
    v_inputs = tuple(f[v_idx_t] for f in feats)

    for ep in range(CONFIG["epochs"]):
        net.train()
        perm = torch.randperm(len(train_idx), device=DEVICE)
        epoch_loss = 0.0; nbatch = 0
        for i in range(0, len(train_idx), SEG_L14_BATCH):
            b = perm[i : i + SEG_L14_BATCH]
            inputs = tuple(f[tr_idx_t[b]] for f in feats)
            opt.zero_grad()
            logits = net(*inputs)
            loss = loss_fn(logits.permute(0, 2, 1), yt[b])
            loss.backward(); opt.step()
            epoch_loss += loss.item(); nbatch += 1

        net.eval()
        with torch.no_grad():
            logits = net(*v_inputs)
            preds  = logits.argmax(dim=-1).cpu().numpy().reshape(-1)
        ious, miou = compute_iou(preds, yv_np)

        log = {"epoch": ep, "train/loss": epoch_loss / nbatch, "val/mIoU": miou}
        for c, v in zip(CLASSES, ious):
            log[f"val/IoU_{c}"] = float(v)
        wandb.log(log)

        if miou > best["val_mIoU"]:
            best = {"val_mIoU": miou,
                    "val_iou_per_class": [float(x) for x in ious],
                    "epoch": ep}

    wandb.finish()
    return best



In [ ]:
# %% CELL 3 - Sweep: full L/14 seg matrix, 63 runs
l14_seg_results = {}
total = 0
for seed in CONFIG["seeds"]:
    print(f"\n##### L/14 SEG SEED {seed} #####")
    l14_seg_results.setdefault(("image_only", "none"), []).append(
        train_seg_l14("image_only", None, seed))
    total += 1
    for cap in CONFIG["captions"]:
        for fusion in ("late", "film", "gated", "cross_attn"):
            l14_seg_results.setdefault((fusion, cap), []).append(
                train_seg_l14(fusion, cap, seed))
            total += 1
print(f"\nTotal L/14 seg runs: {total}")




##### L/14 SEG SEED 42 #####


epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train/loss,█▄▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/IoU_Barren,▁▁▂▄▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇▆▇▇██▇█▇██
val/IoU_Built-up,▁▁▂▄▅▆▆▇▇▇▇▇▇▇████████████████
val/IoU_Crop,▁▂▄▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇██▇▇███
val/IoU_Grass,▁▃▄▄▅▅▅▆▆▆▆▆▆▇▇▆▇▇▇▇▇█████████
val/IoU_Shrub,▁▁▁▁▁▁▁▁▁▁▁▁▂▃▃▅▅▄▄▅▆▆▇▆▅▇█▆▅█
val/IoU_Tree,▁▃▄▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇███████
val/IoU_Water,▁▂▆▇▇▇▇▇██████████████████████
val/mIoU,▁▂▄▅▆▆▆▇▇▇▇▇▇▇▇▇▇█▇███████████
epoch,29


epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train/loss,█▄▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/IoU_Barren,▁▅▆▇▇▇▇▇▇▇▇▇█▇██▇▇████████████
val/IoU_Built-up,▁▁▅▆▇▇▇▇▇▇▇▇▇█████████████████
val/IoU_Crop,▁▄▅▅▅▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇████████
val/IoU_Grass,▁▃▅▅▆▅▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇███▇█████
val/IoU_Shrub,▁▁▁▁▁▁▁▁▂▂▃▃▃▆▅▆▆▆▇▅▆▇▆▇▇▇▇▇██
val/IoU_Tree,▁▄▅▅▆▆▆▆▆▆▇▇▆▇▇▇▇▇▇▇██▇███████
val/IoU_Water,▁▆▆▇▇▇▇███████████████████████
val/mIoU,▁▄▅▆▆▆▆▇▇▇▇▇▇▇▇██▇████████████
epoch,29


epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train/loss,█▄▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/IoU_Barren,▁▆▆▆▆▇▇▇▇▇█▇█▇██▇▇███▇████████
val/IoU_Built-up,▁▄▇▇▇▇▇▇▇█▇█▇█████████████████
val/IoU_Crop,▁▁▅▆▆▆▇▅▆▇▆▇▇█▇▇██▇███████▇█▇█
val/IoU_Grass,▁▃▄▅▆▆▆▇▇▇▇▇▇▇▇▇▇█▇▇▇▇████████
val/IoU_Shrub,▁▁▁▁▆▄▂▅▃▄▄▆▅▅▅▅▅▆▇█▆▇▅▆▇▆██▇▆
val/IoU_Tree,▁▃▅▅▆▆▇▆▇▇▇▇▇▇█▇▇█████▇███▇███
val/IoU_Water,▁▅▆▆▆▇▇▇▇▇▇█▆█████████████████
val/mIoU,▁▄▅▆▆▇▆▇▇▇▇▇▇▇▇▇▇████▇████████
epoch,29


epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train/loss,█▄▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/IoU_Barren,▁▄▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇████████████
val/IoU_Built-up,▁▁▁▄▅▆▆▇▇▇▇▇▇▇████████████████
val/IoU_Crop,▁▃▅▅▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇█▇████████
val/IoU_Grass,▁▄▅▆▆▆▆▇▇▇▇▇▇▇▇▇██████████████
val/IoU_Shrub,▁▁▁▁▁▁▁▁▁▁▁▁▂▃▃▅▅▅▆▅▆▇▆▇▇▇██▇█
val/IoU_Tree,▁▄▅▆▆▆▇▇▇▇▇▇▇▇▇▇██████████████
val/IoU_Water,▁▅▆▆▇▇▇▇██████████████████████
val/mIoU,▁▃▄▅▆▆▆▇▇▇▇▇▇▇▇▇▇▇████████████
epoch,29


...

epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train/loss,█▄▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁
val/IoU_Barren,▁▁▄▅▅▇▅▆▆▇▆▆█▇▆▇▇▇▇█▆█▇▇▇█▇▆█▇
val/IoU_Built-up,▁▁▅▆▆▆▇▇▇▇▇▇██▇▇██████████████
val/IoU_Crop,▁▃▃▄▅▅▆▆▅▆▆▅▆▇▇▇▇▇▇▇▇▇█▇██▇███
val/IoU_Grass,▁▃▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇██████
val/IoU_Shrub,▁▁▁▁▁▁▁▁▁▁▁▁▂▂▁▂▃▃▃▂▄▅▄█▅▄▄▄▇█
val/IoU_Tree,▁▄▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇▇████████████
val/IoU_Water,▁▆▇▇▇▇▇▇▇█████████████████████
val/mIoU,▁▃▅▆▆▇▆▇▇▇▇▇▇▇▇▇▇▇▇█▇█████████
epoch,29


epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train/loss,█▅▄▃▃▃▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁
val/IoU_Barren,▁▃▄▅▅▆▅▆▇▆▇▆▇▇▇▄▅▆▇▆▆█▇▇▇▇▇█▇▇
val/IoU_Built-up,▁▅▆▇▇▇█████▇███▆██████████████
val/IoU_Crop,▂▁▅▅▅▆▇▇▇▇▇█▇▇█▇▇▇▆▇████▇▇██▇█
val/IoU_Grass,▁▄▅▆▅▇▇▇▇█▇▇███▇██▇██▇█▇██████
val/IoU_Shrub,▁▁▁▁▂▂▄▂▄█▄▂▃▅▃▄▆▆█▄▄▄▆█▅█▅▆█▇
val/IoU_Tree,▁▄▁▆▆▇▇▆▇▇▇▇▇██▇███▇▇▇█▇██████
val/IoU_Water,▁▅▆▅▇▇▇▇▇▇█▇██████████████████
val/mIoU,▁▄▅▅▆▇▇▇▇█▇▇▇██▆▇██▇▇█████████
epoch,29


epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train/loss,█▅▄▃▃▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁
val/IoU_Barren,▁▁▁▃▄▆▅▆▆▆▆▆▇▇▇▇▇▇▇█▇█▇▇▇▇▇▇██
val/IoU_Built-up,▁▁▁▃▅▆▇▇▇▇▇▇▇█▇███████████████
val/IoU_Crop,▁▃▄▅▆▆▆▇▆▇▇▆▇▇▇▇▇█▇███████████
val/IoU_Grass,▁▃▅▆▆▆▆▇▇▇▇▇▇▇▇▇▇████▇████████
val/IoU_Shrub,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▂▃▂▃▄▄▇▄▅▄▄▇█
val/IoU_Tree,▁▄▅▆▆▆▇▇▇▇▇▇▇▇▇███████████████
val/IoU_Water,▁▄▆▆▇▇▇▇▇█████████████████████
val/mIoU,▁▃▄▅▆▆▆▇▇▇▇▇▇▇▇▇▇█████████████
epoch,29


epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train/loss,█▅▄▃▃▃▂▂▂▂▂▂▂▂▂▂▂▁▁▂▁▁▁▁▁▁▁▁▁▁
val/IoU_Barren,▂▁▆▁▅▆▆▇▆▇▇▇▅█▇█▆█▇▇▇▆▇▆▅▇██▆▇
val/IoU_Built-up,▁▄▅▄▆▇█▇▇█▇██████▇█▇█▆▇▇▇▇███▅
val/IoU_Crop,▁▃▄▅▅▆▇▆▇█▆▇▇▅█▇▇▇▇▆▇▇█▇▇▇█▇▆█
val/IoU_Grass,▁▄▅▅▇▃▇█▇▆▆▆▇█▇█▆▇█▆▆█▇██▇▅█▇█
val/IoU_Shrub,▁▂▂▂▂▇▅▄▂▄▆▇▆▇▃▄▅▂▃▆▆▃▃▃▃▄▃▆▄█
val/IoU_Tree,▁▄▄▅▅▆▇▇█▇▆▇█▇██▆██▇██▇███▇▇▇█
val/IoU_Water,▁▃▆▄▇▆▇▇█▇▇▇██▇██▇▆▇███▇██▇▇▇█
val/mIoU,▁▂▅▃▆▇▇▇▇▇▇█▇█▇█▇▇▇▇█▇▇▇▆▇▇█▇█
epoch,29



Total L/14 seg runs: 63


In [ ]:
# %% CELL 4 - Aggregate + save + headline tables (vs Phase-2/B/32 seg numbers)
l14_seg_agg = {}
for (fusion, cap), runs in l14_seg_results.items():
    miou = np.array([r["val_mIoU"] for r in runs])
    iouc = np.array([r["val_iou_per_class"] for r in runs])
    l14_seg_agg[f"{fusion}__{cap}"] = {
        "fusion": fusion, "caption": cap, "n_seeds": len(runs),
        "mIoU_mean": float(miou.mean()), "mIoU_std": float(miou.std()),
        "iou_per_class_mean": iouc.mean(0).tolist(),
        "iou_per_class_std":  iouc.std(0).tolist(),
        "backbone": "RemoteCLIP-ViT-L/14",
    }

out_path = CONFIG["results_dir"] / "seg_l14.json"
with open(out_path, "w") as f:
    json.dump(l14_seg_agg, f, indent=2)
print(f"Saved: {out_path}")

print("\n=== L/14 seg mIoU heatmap (3-seed mean) ===")
img = l14_seg_agg["image_only__none"]
print(f"  image-only: {img['mIoU_mean']:.3f}+/-{img['mIoU_std']:.3f}")
print(f"{'fusion':12s}  " + "  ".join(f"{c[:14]:>14s}" for c in CONFIG["captions"]) + "       mean")
for fusion in ("late", "film", "gated", "cross_attn"):
    row = [l14_seg_agg[f"{fusion}__{cap}"]["mIoU_mean"] for cap in CONFIG["captions"]]
    print(f"{fusion:12s}  " + "  ".join(f"{m:>14.3f}" for m in row) + f"  {sum(row)/len(row):.3f}")

print("\n=== Per-class IoU lift: best CA (L/14) vs image-only (L/14) ===")
best_ca_key = max((k for k, v in l14_seg_agg.items() if v["fusion"] == "cross_attn"),
                  key=lambda k: l14_seg_agg[k]["mIoU_mean"])
best_ca = l14_seg_agg[best_ca_key]
print(f"  Best CA: {best_ca_key}  mIoU {best_ca['mIoU_mean']:.3f}+/-{best_ca['mIoU_std']:.3f}")
for c, base_m, ca_m in zip(CLASSES, img["iou_per_class_mean"], best_ca["iou_per_class_mean"]):
    print(f"    {c:10s}  image-only {base_m:.3f}  ->  best-CA {ca_m:.3f}   delta={ca_m - base_m:+.3f}")


Saved: /content/drive/MyDrive/Colab Notebooks/DI725/phase3_results/seg_l14.json

=== L/14 seg mIoU heatmap (3-seed mean) ===
  image-only: 0.533+/-0.002
fusion        hybrid_gemma3-  hybrid_qwen3-v   text_qwen3-4b  vision_gemma3-  vision_qwen3-v       mean
late                   0.623           0.627           0.635           0.536           0.535  0.591
film                   0.648           0.654           0.657           0.570           0.568  0.619
gated                  0.621           0.626           0.638           0.540           0.539  0.593
cross_attn             0.669           0.670           0.658           0.562           0.561  0.624

=== Per-class IoU lift: best CA (L/14) vs image-only (L/14) ===
  Best CA: cross_attn__hybrid_qwen3-vl-8b  mIoU 0.670+/-0.002
    Tree        image-only 0.760  ->  best-CA 0.814   delta=+0.054
    Shrub       image-only 0.034  ->  best-CA 0.378   delta=+0.344
    Grass       image-only 0.715  ->  best-CA 0.793   delta=+0.077
    Crop       

## 04 Example Samples

This cell builds a full-width figure with 3 example val
images, each row showing:
  - input image (224x224)
  - GT segmentation mask (224x224 colour-coded)
  - 5 caption excerpts side by side (truncated)
  - best-CA prediction overlay (L/14, 16x16 grid)

Image selection: one dominant-class scene, one balanced multi-class, one with a minor class present


In [2]:
# %% CELL 1 - Pick 3 val images covering the dominant / balanced / minor regime
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from PIL import Image

for g in ("CONFIG", "df", "val_idx", "set_seeds",
          "CASeg", "image_patches_l14_gpu", "text_tokens_l14",
          "patch_labels_l14_gpu", "L14_GRID", "L14_FEAT_DIM"):
    assert g in globals(), f"{g} missing - run upstream cells first."

CLASS_COLORS = np.array([
    [0,   100, 0],     # 0 Tree
    [255, 182, 193],   # 1 Shrub
    [154, 205, 50],    # 2 Grass
    [255, 215, 0],     # 3 Crop
    [139, 69,  19],    # 4 Built-up
    [211, 211, 211],   # 5 Barren
    [0,   0,   255],   # 6 Water
], dtype=np.int32)
PALETTE = CLASS_COLORS / 255.0


def select_three_samples():
    """Pick three distinct val indices: (a) one dominant class, (b) balanced
    multi-class, (c) at least one minor class (Shrub / Built-up / Water) present.
    Later picks skip any val index already chosen by earlier picks."""
    rows = df.loc[val_idx, CLASSES].values        # [Nv, 7] composition %
    chosen = []

    # (a) dominant: one class >= 80%
    dominant_mask = (rows.max(1) >= 80)
    pos = np.where(dominant_mask)[0]
    chosen.append(int(val_idx[pos[0]]) if len(pos) else int(val_idx[0]))

    # (b) balanced: top class < 50% and 3+ classes >= 10%, distinct from (a)
    n_present = (rows >= 10).sum(1)
    balanced_mask = (rows.max(1) < 50) & (n_present >= 3)
    for i in np.where(balanced_mask)[0]:
        if int(val_idx[i]) not in chosen:
            chosen.append(int(val_idx[i])); break
    else:
        chosen.append(int(val_idx[1]) if int(val_idx[1]) not in chosen else int(val_idx[2]))

    # (c) minor: Shrub/Built-up/Water composition >= 10%, distinct from (a) and (b)
    minor_classes = [CLASSES.index(c) for c in ("Shrub", "Built-up", "Water")]
    minor_mask = (rows[:, minor_classes] >= 10).any(1)
    for i in np.where(minor_mask)[0]:
        if int(val_idx[i]) not in chosen:
            chosen.append(int(val_idx[i])); break
    else:
        # fallback: take any val image not already chosen
        for i in range(len(val_idx)):
            if int(val_idx[i]) not in chosen:
                chosen.append(int(val_idx[i])); break

    assert len(set(chosen)) == 3, f"Failed to pick three distinct samples: {chosen}"
    return chosen


picks = select_three_samples()
print("Picked val indices:")
for vi in picks:
    comp = df.loc[vi, CLASSES].to_dict()
    nonzero = {c: round(v, 1) for c, v in comp.items() if v >= 5}
    print(f"  idx={vi}  filename={df['filename'].iloc[vi]}  composition>=5%: {nonzero}")



Picked val indices:
  idx=5535  filename=248125.png  composition>=5%: {'Tree': 7, 'Grass': 6, 'Crop': 85}
  idx=9446  filename=85147.png  composition>=5%: {'Tree': 39, 'Shrub': 33, 'Grass': 28}
  idx=9581  filename=88121.png  composition>=5%: {'Tree': 59, 'Shrub': 19, 'Grass': 21}


In [3]:
# %% CELL 2 - Train one CA-seg model at seed 42 with the report's best caption
EX_CAPTION = "hybrid_qwen3-vl-8b"   # the consistent winner in Phase-2 and 03c
EX_SEED    = 42

set_seeds(EX_SEED)
text_t_l14 = text_tokens_l14[EX_CAPTION].to(DEVICE).float()
viz_net = CASeg(dim=L14_FEAT_DIM).to(DEVICE)
opt = torch.optim.AdamW(viz_net.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
loss_fn = nn.CrossEntropyLoss()

yt = patch_labels_l14_gpu[train_idx]
tr_idx_t = torch.tensor(train_idx, device=DEVICE)
EX_BATCH = 64

for ep in range(CONFIG["epochs"]):
    viz_net.train()
    perm = torch.randperm(len(train_idx), device=DEVICE)
    for i in range(0, len(train_idx), EX_BATCH):
        b = perm[i : i + EX_BATCH]
        idx_b = tr_idx_t[b]
        opt.zero_grad()
        logits = viz_net(image_patches_l14_gpu[idx_b].float(), text_t_l14[idx_b])
        loss = loss_fn(logits.permute(0, 2, 1), yt[b])
        loss.backward(); opt.step()

viz_net.eval()
with torch.no_grad():
    v_idx_t = torch.tensor(picks, device=DEVICE)
    logits = viz_net(image_patches_l14_gpu[v_idx_t].float(), text_t_l14[v_idx_t])
    preds = logits.argmax(dim=-1).cpu().numpy()   # [3, 196]
print(f"Generated predictions for {len(picks)} examples.")



Generated predictions for 3 examples.


In [4]:
# %% CELL 3 - Render the example-samples figure
def truncate(text, n_words=14):
    if not isinstance(text, str):
        return ""
    words = text.split()
    return " ".join(words[:n_words]) + ("..." if len(words) > n_words else "")


fig = plt.figure(figsize=(16, 9), constrained_layout=True)
gs  = fig.add_gridspec(3, 8, width_ratios=[1, 1, 1, 1, 1, 1, 1, 1])

for row, vi in enumerate(picks):
    name = df["filename"].iloc[vi]
    img      = np.array(Image.open(CONFIG["data_root"] / "images" / name).convert("RGB"))
    mask_rgb = np.array(Image.open(CONFIG["data_root"] / "masks"  / name).resize((224, 224), Image.NEAREST))
    pred_grid = preds[row].reshape(L14_GRID, L14_GRID)
    pred_rgb  = (PALETTE[pred_grid] * 255).astype(np.uint8)

    ax_img  = fig.add_subplot(gs[row, 0])
    ax_gt   = fig.add_subplot(gs[row, 1])
    ax_pred = fig.add_subplot(gs[row, 2])
    ax_img.imshow(img);                  ax_img.set_title(f"image idx={vi}",  fontsize=9); ax_img.axis("off")
    ax_gt.imshow(mask_rgb);              ax_gt.set_title("GT mask",            fontsize=9); ax_gt.axis("off")
    ax_pred.imshow(pred_rgb);            ax_pred.set_title("Best-CA pred (L/14)", fontsize=9); ax_pred.axis("off")

    # 5 caption excerpts as text boxes
    cap_ax = fig.add_subplot(gs[row, 3:])
    cap_ax.axis("off")
    lines = []
    for col in CONFIG["captions"]:
        text = truncate(df[col].iloc[vi])
        lines.append(f"[{col}]  {text}")
    cap_ax.text(0.0, 0.5, "\n".join(lines), fontsize=8, family="monospace",
                verticalalignment="center")

# Class legend
legend_handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in PALETTE]
fig.legend(legend_handles, CLASSES, loc="lower center", ncol=len(CLASSES),
           bbox_to_anchor=(0.5, -0.02), fontsize=9, frameon=False)

out_path = CONFIG["results_dir"] / "fig_example_samples.png"
fig.savefig(out_path, dpi=300, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {out_path}")


Saved: /content/drive/MyDrive/Colab Notebooks/DI725/phase3_results/fig_example_samples.png


## 05 Phase3 Figures

Analysis figures for the Phase-3 ablation section
Three publication PNGs at 300 dpi, B&W-readable (hatch patterns + value
annotations + viridis):
  fig_tau_sensitivity.png  - tau (5/10/20%) heatmap: fusion family rows
                             vs tau columns (mAP), with overlay numbers.
  fig_l14_vs_b32_seg.png   - bar chart per fusion: mIoU at B/32 vs L/14,
                             hatched per backbone. Headline 'severely
                             affects results' ablation figure.
  fig_perclass_l14.png     - per-class IoU lift (image-only vs best CA)
                             for L/14 seg, side-by-side bars + delta line.




In [ ]:
# %% CELL 1 - Load all JSON results from disk (decoupled from in-memory state)
import json

import numpy as np
import matplotlib.pyplot as plt

assert "CONFIG" in globals() and "CLASSES" in globals(), "Run 01 CELL 1 first."

RESULTS_DIR = CONFIG["results_dir"]
FIG_DIR     = RESULTS_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

def load(path):
    with open(path) as f:
        return json.load(f)

tau_agg      = load(RESULTS_DIR / "tau_ablation.json")
seg_b32_agg  = load(RESULTS_DIR / "seg_full_b32.json")
pathA_l14_agg = load(RESULTS_DIR / "pathA_l14.json")   # not plotted here but loaded for sanity
seg_l14_agg  = load(RESULTS_DIR / "seg_l14.json")

print(f"tau_ablation entries: {len(tau_agg)}")
print(f"seg_b32 entries:      {len(seg_b32_agg)}")
print(f"pathA_l14 entries:    {len(pathA_l14_agg)}")
print(f"seg_l14 entries:      {len(seg_l14_agg)}")



tau_ablation entries: 63
seg_b32 entries:      21
pathA_l14 entries:    21
seg_l14 entries:      21


In [ ]:
# %% CELL 2 - Figure: tau sensitivity heatmap (rows = fusion family, cols = tau, value = best mAP)
TAUS    = CONFIG["taus"]
FUSIONS = ("image_only", "late", "film", "gated", "cross_attn")
CAPTIONS = CONFIG["captions"]


def best_mAP_for(fusion, tau):
    """Best mAP across captions for a given fusion family at a given tau."""
    if fusion == "image_only":
        v = tau_agg[f"image_only__none__tau{tau}"]
        return v["mAP_mean"], v["mAP_std"]
    keys = [k for k, v in tau_agg.items()
            if v["fusion"] == fusion and v["tau"] == tau]
    best_k = max(keys, key=lambda k: tau_agg[k]["mAP_mean"])
    v = tau_agg[best_k]
    return v["mAP_mean"], v["mAP_std"]


grid = np.array([[best_mAP_for(f, t)[0] for t in TAUS] for f in FUSIONS])

fig, ax = plt.subplots(figsize=(5.5, 3.6))
im = ax.imshow(grid, cmap="viridis", aspect="auto", vmin=grid.min() - 0.02, vmax=grid.max() + 0.02)
ax.set_xticks(range(len(TAUS)));     ax.set_xticklabels([f"tau={t}%" for t in TAUS])
ax.set_yticks(range(len(FUSIONS)));  ax.set_yticklabels([f for f in FUSIONS])
for i, f in enumerate(FUSIONS):
    for j, t in enumerate(TAUS):
        m, s = best_mAP_for(f, t)
        ax.text(j, i, f"{m:.3f}\n+/-{s:.3f}", ha="center", va="center", fontsize=8,
                color="white" if grid[i, j] < (grid.min() + grid.max()) / 2 else "black")
ax.set_title("Best mAP per fusion family across tau (Path A)", fontsize=10)
fig.colorbar(im, ax=ax, label="mAP")
fig.tight_layout()
out = FIG_DIR / "fig_tau_sensitivity.png"
fig.savefig(out, dpi=300, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {out}")



Saved: /content/drive/MyDrive/Colab Notebooks/DI725/phase3_results/figures/fig_tau_sensitivity.png


In [ ]:
# %% CELL 3 - Figure: L/14 vs B/32 segmentation mIoU per fusion family (hatched bars)
def best_mIoU(agg_dict, fusion):
    if fusion == "image_only":
        v = agg_dict["image_only__none"]
        return v["mIoU_mean"], v["mIoU_std"]
    keys = [k for k, v in agg_dict.items() if v["fusion"] == fusion]
    best_k = max(keys, key=lambda k: agg_dict[k]["mIoU_mean"])
    v = agg_dict[best_k]
    return v["mIoU_mean"], v["mIoU_std"]


b32 = [best_mIoU(seg_b32_agg, f) for f in FUSIONS]
l14 = [best_mIoU(seg_l14_agg, f) for f in FUSIONS]

x = np.arange(len(FUSIONS))
width = 0.36

fig, ax = plt.subplots(figsize=(6.4, 3.6))
bars32 = ax.bar(x - width/2, [m for m, _ in b32], width, yerr=[s for _, s in b32],
                label="B/32 (7x7)", color="#4c72b0", edgecolor="black", hatch="//", capsize=3)
bars16 = ax.bar(x + width/2, [m for m, _ in l14], width, yerr=[s for _, s in l14],
                label="L/14 (16x16)", color="#dd8452", edgecolor="black", hatch="\\\\", capsize=3)

for bar, (m, _) in zip(bars32, b32):
    ax.text(bar.get_x() + bar.get_width()/2, m + 0.005, f"{m:.3f}",
            ha="center", va="bottom", fontsize=7)
for bar, (m, _) in zip(bars16, l14):
    ax.text(bar.get_x() + bar.get_width()/2, m + 0.005, f"{m:.3f}",
            ha="center", va="bottom", fontsize=7)

ax.set_xticks(x);  ax.set_xticklabels(FUSIONS, rotation=15)
ax.set_ylabel("mIoU (best caption per family, 3-seed mean)")
ax.set_title("Backbone resolution ablation: B/32 vs L/14 segmentation", fontsize=10)
ax.set_ylim(0, max(max(m for m, _ in l14), max(m for m, _ in b32)) + 0.08)
ax.legend(loc="lower right")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
out = FIG_DIR / "fig_l14_vs_b32_seg.png"
fig.savefig(out, dpi=300, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {out}")



Saved: /content/drive/MyDrive/Colab Notebooks/DI725/phase3_results/figures/fig_l14_vs_b32_seg.png


In [ ]:
# %% CELL 4 - Figure: per-class IoU lift image-only -> best CA, at L/14
img = seg_l14_agg["image_only__none"]
ca_keys = [k for k, v in seg_l14_agg.items() if v["fusion"] == "cross_attn"]
best_ca = seg_l14_agg[max(ca_keys, key=lambda k: seg_l14_agg[k]["mIoU_mean"])]

img_per_class = np.array(img["iou_per_class_mean"])
ca_per_class  = np.array(best_ca["iou_per_class_mean"])
delta         = ca_per_class - img_per_class

x = np.arange(len(CLASSES))
width = 0.36

fig, ax = plt.subplots(figsize=(7.5, 3.8))
b1 = ax.bar(x - width/2, img_per_class, width,
            label="image-only seg (L/14)", color="#999999", edgecolor="black", hatch="//")
b2 = ax.bar(x + width/2, ca_per_class, width,
            label=f"best CA seg (L/14, {best_ca['caption']})",
            color="#2ca02c", edgecolor="black", hatch="\\\\")

for bars, vals in ((b1, img_per_class), (b2, ca_per_class)):
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.01, f"{v:.2f}",
                ha="center", va="bottom", fontsize=7)
for xi, d in zip(x, delta):
    ax.annotate(f"+{d:.2f}" if d >= 0 else f"{d:.2f}",
                xy=(xi, max(img_per_class[xi], ca_per_class[xi]) + 0.07),
                ha="center", fontsize=8, color="red")

ax.set_xticks(x);  ax.set_xticklabels(CLASSES, rotation=15)
ax.set_ylabel("IoU (3-seed mean)")
ax.set_title(f"Per-class IoU lift at L/14: image-only vs best CA  "
             f"(mIoU {img['mIoU_mean']:.3f} -> {best_ca['mIoU_mean']:.3f})", fontsize=10)
ax.set_ylim(0, 1.05)
ax.legend(loc="upper right")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
out = FIG_DIR / "fig_perclass_l14.png"
fig.savefig(out, dpi=300, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {out}")



Saved: /content/drive/MyDrive/Colab Notebooks/DI725/phase3_results/figures/fig_perclass_l14.png


In [ ]:
# %% CELL 5 - Cross-task summary table (printed; copy into the report)
print("\n=== Cross-task summary (Phase-3 numbers) ===")
print(f"{'metric':40s}  {'value':>20s}")
print("-" * 65)

# Path A baseline + best CA at tau=10
img_pA = tau_agg["image_only__none__tau10"]
ca_keys_pA = [k for k, v in tau_agg.items() if v["fusion"] == "cross_attn" and v["tau"] == 10]
best_ca_pA = tau_agg[max(ca_keys_pA, key=lambda k: tau_agg[k]["mAP_mean"])]
print(f"{'Path A image-only mAP (tau=10)':40s}  {img_pA['mAP_mean']:.3f}+/-{img_pA['mAP_std']:.3f}")
print(f"{'Path A best CA mAP (tau=10)':40s}  {best_ca_pA['mAP_mean']:.3f}+/-{best_ca_pA['mAP_std']:.3f}")
print(f"{'Path A CA-vs-baseline delta':40s}  +{best_ca_pA['mAP_mean'] - img_pA['mAP_mean']:.3f}")

# Path B B/32 baseline + best CA
img_b32 = seg_b32_agg["image_only__none"]
ca_keys_b32 = [k for k, v in seg_b32_agg.items() if v["fusion"] == "cross_attn"]
best_ca_b32 = seg_b32_agg[max(ca_keys_b32, key=lambda k: seg_b32_agg[k]["mIoU_mean"])]
print(f"{'Path B B/32 image-only mIoU':40s}  {img_b32['mIoU_mean']:.3f}+/-{img_b32['mIoU_std']:.3f}")
print(f"{'Path B B/32 best CA mIoU':40s}  {best_ca_b32['mIoU_mean']:.3f}+/-{best_ca_b32['mIoU_std']:.3f}")

# Path B L/14
print(f"{'Path B L/14 image-only mIoU':40s}  {img['mIoU_mean']:.3f}+/-{img['mIoU_std']:.3f}")
print(f"{'Path B L/14 best CA mIoU':40s}  {best_ca['mIoU_mean']:.3f}+/-{best_ca['mIoU_std']:.3f}")
print(f"{'Path B L/14 vs B/32 best-CA delta':40s}  "
      f"+{best_ca['mIoU_mean'] - best_ca_b32['mIoU_mean']:.3f}")



=== Cross-task summary (Phase-3 numbers) ===
metric                                                   value
-----------------------------------------------------------------
Path A image-only mAP (tau=10)            0.817+/-0.001
Path A best CA mAP (tau=10)               0.948+/-0.003
Path A CA-vs-baseline delta               +0.131
Path B B/32 image-only mIoU               0.574+/-0.003
Path B B/32 best CA mIoU                  0.711+/-0.003
Path B L/14 image-only mIoU               0.533+/-0.002
Path B L/14 best CA mIoU                  0.670+/-0.002
Path B L/14 vs B/32 best-CA delta         +-0.042


In [ ]:
# Bundle Phase-2 JSONs + Phase-3 results, download
import shutil
from pathlib import Path
from google.colab import files

DRIVE = Path("/content/drive/MyDrive/Colab Notebooks/DI725")
STAGE = Path("/content/phase3_bundle")
if STAGE.exists():
    shutil.rmtree(STAGE)
STAGE.mkdir(parents=True)

shutil.copytree(DRIVE / "phase3_results", STAGE / "phase3_results")

ph2_dest = STAGE / "phase2_results"
ph2_dest.mkdir()
for j in (DRIVE / "phase2_features").glob("*.json"):
    shutil.copy(j, ph2_dest / j.name)
    print(f"  + {j.name}")

zip_path = shutil.make_archive("/content/phase3_bundle", "zip", STAGE)
print(f"\nZipped -> {zip_path}  ({Path(zip_path).stat().st_size / 1e6:.1f} MB)")
files.download(zip_path)


  + pathA_results.json
  + pathA_results_multi_seed.json
  + pathA_backbone_ablation.json
  + pathA_fusion_variants.json
  + pathB_seg_results.json

Zipped -> /content/phase3_bundle.zip  (2.8 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>